# Prerequisite-Aware Spreading-Activation GraphRAG for Pre-Calculus

This notebook extends the original pre-calculus RAG pipeline into a **knowledge-graph-based RAG** inspired by:

1. **SA-RAG**: a heterogeneous graph indexing + subgraph fetching + spreading activation retrieval pipeline.  
2. **PREREQ**: concept prerequisite inference from educational resources, using latent concept representations and pairwise prediction.  
3. **A new proposal** introduced here: **PEP-SA-RAG** (**P**rerequisite-**E**nhanced **P**opulation-dynamics Spreading Activation RAG), designed for mathematics education.

## What this notebook does

- reuses the original book preprocessing logic
- builds chunks, embeddings, dense retrieval, and a heterogeneous graph
- infers **concept prerequisite edges**
- implements **spreading activation retrieval**
- adds a **population-dynamics re-ranking step** over activated nodes
- compares several retrieval strategies
- evaluates multiple generators:
  - **OpenAI**
  - **Gemini**
  - **DeepSeek**
  - **open-source models** such as **Phi-4**, **Llama 3.1 Instruct**, and **Qwen2.5 Instruct**

## Recommended evaluation modes

### Option A: Your pre-calculus book
Use the uploaded book / local PDF as the knowledge base.  
This is better for your doctoral proposal because it matches your educational domain.

### Option B: Public multi-hop QA benchmarks
Use benchmark datasets such as MuSiQue or 2WikiMultiHopQA to compare against published GraphRAG-style baselines.  
This is better for more standardized comparison, but it is less aligned with mathematics education.

This notebook is organized so that you can run either option.

---

In [40]:
# ============================================================
# 0. Installation
# ============================================================
# Uncomment if needed in a clean Colab environment.

# !pip install -q pymupdf pandas pyarrow numpy scipy scikit-learn \
#     networkx faiss-cpu sentence-transformers rank-bm25 matplotlib seaborn tqdm \
#     unidecode pydantic rapidfuzz openai google-generativeai transformers \
#     accelerate bitsandbytes torch evaluate datasets

# If you are in Google Colab:
# from google.colab import drive
# drive.mount("/content/drive")


In [41]:
#from google.colab import drive
#drive.mount("/content/drive")


In [10]:
# ============================================================
# 🔥 LOCAL SETUP (Ambiente Local - No Colab)
# ============================================================
# All packages should already be installed via pip/venv

# ============================================================
# 📦 NLTK
# ============================================================
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

# ============================================================
# ⚠️ EVITAR BUGS DE TRANSFORMERS
# ============================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("✅ Setup completo (ambiente local)")

✅ Setup completo (ambiente local)


In [12]:
# 🔥 REINICIO AUTOMÁTICO (NO NECESARIO EN AMBIENTE LOCAL)
# import os
# os.kill(os.getpid(), 9)
print("Kernel restart skipped in local environment")

Kernel restart skipped in local environment


In [13]:

# ============================================================
# 1. Imports
# ============================================================
import os
import re
import gc
import json
import math
import time
import pickle
import random
import textwrap
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import fitz
import faiss
import networkx as nx
import matplotlib.pyplot as plt

from unidecode import unidecode
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import minmax_scale
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
random.seed(42)
np.random.seed(42)

## 2. Configuration

This section centralizes all paths, model names, and experiment parameters.

The notebook keeps the **same preprocessing spirit** as your original notebook:
- page extraction
- cleaning
- chapter/section structure
- word-based chunking
- embedding creation
- FAISS indexing

Then it adds:
- graph construction
- prerequisite inference
- SA retrieval
- population-dynamics re-ranking
- multi-LLM evaluation

In [14]:
# ============================================================
# 2. Global configuration
# ============================================================
# Main project paths - adapted for local execution
import os
from pathlib import Path

# Get the directory of this notebook
NOTEBOOK_DIR = Path("/Users/camilacruz/Documents/repositorios/ai_tutor/notebooks")

# Set up project directories relative to the notebook location
BOOK_DIR = NOTEBOOK_DIR.parent / "doctorado" / "libro"      # folder containing the book PDF
OUTPUT_DIR = NOTEBOOK_DIR.parent / "doctorado" / "libro" / "output"   # everything will be saved here

# Optional: exact PDF name. If left as None, the notebook will automatically
# search BOOK_DIR for the first PDF file it finds.
BOOK_FILENAME = None   # e.g. "precalculus_book.pdf" or "libro.pdf"

# Choose whether to run on the local pre-calculus book or on an external benchmark corpus
USE_LOCAL_BOOK = True
BENCHMARK_DIR = NOTEBOOK_DIR.parent / "doctorado" / "benchmarks"

BOOK_DIR_PATH = Path(BOOK_DIR)
OUTPUT_DIR_PATH = Path(OUTPUT_DIR)

# Research artifact folders inside doctorado/libro
WORKDIR = OUTPUT_DIR_PATH / "pep_sa_rag_project"
RAW_DIR = WORKDIR / "raw"
PROC_DIR = WORKDIR / "processed"
EMB_DIR = WORKDIR / "embeddings"
INDEX_DIR = WORKDIR / "indexes"
GRAPH_DIR = WORKDIR / "graphs"
EVAL_DIR = WORKDIR / "eval"
PLOT_DIR = WORKDIR / "plots"
TABLE_DIR = WORKDIR / "tables"
CACHE_DIR = WORKDIR / "cache"
LOG_DIR = WORKDIR / "logs"

for d in [WORKDIR, RAW_DIR, PROC_DIR, EMB_DIR, INDEX_DIR, GRAPH_DIR, EVAL_DIR, PLOT_DIR, TABLE_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def discover_book_path(book_dir: Path, book_filename: Optional[str] = None) -> Path:
    if book_filename:
        candidate = book_dir / book_filename
        if candidate.exists():
            return candidate
        raise FileNotFoundError(f"BOOK_FILENAME was provided but not found: {candidate}")

    # Create book_dir if it doesn't exist
    book_dir.mkdir(parents=True, exist_ok=True)
    
    pdfs = sorted(book_dir.glob("*.pdf"))
    if len(pdfs) == 0:
        print(f"⚠️  No PDF files found in {book_dir}")
        print(f"Please place a pre-calculus book PDF in: {book_dir}")
        return None
    if len(pdfs) > 1:
        print("Multiple PDFs found in BOOK_DIR. Using the first one:", pdfs[0].name)
    return pdfs[0]

BOOK_PATH = discover_book_path(BOOK_DIR_PATH, BOOK_FILENAME)

# ---------- text preprocessing ----------
CHUNK_SIZE_WORDS = 350
CHUNK_OVERLAP_WORDS = 80
MIN_CHUNK_WORDS = 80

# ---------- embedding model ----------
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# ---------- topic / prerequisite model ----------
TOPIC_MODEL_KIND = "nmf"  # "lda" or "nmf"
N_TOPICS = 40

# ---------- graph retrieval ----------
TOPK_DENSE = 8
TOPK_SEED_CONCEPTS = 8
SUBGRAPH_HOPS = 2
SA_EDGE_FLOOR = 0.40
SA_THRESHOLD = 0.50
DOC_SIM_THRESHOLD = 0.20
RELATION_SIM_THRESHOLD = 0.20

# ---------- population dynamics ----------
POP_ITERS = 12
POP_TEMP = 0.8
POP_MIX_LAMBDA = 0.55
POP_PREREQ_ALPHA = 0.60
POP_CENTRALITY_BETA = 0.20
POP_SEMANTIC_GAMMA = 0.20

# ---------- generation ----------
MAX_CONTEXT_CHARS = 9000
MAX_RETRIEVAL_DOCS = 8

# ---------- evaluation ----------
RANDOM_STATE = 42
TEST_SIZE = 0.30

# ---------- LLM providers / models ----------
OPENAI_MODEL = "gpt-4.1-mini"
GEMINI_MODEL = "gemini-2.5-pro"
DEEPSEEK_MODEL = "deepseek-chat"

OPEN_SOURCE_MODELS = [
    "microsoft/Phi-4-mini-instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct"
]

# ---------- Global API keys ----------
# Use environment variables for safety
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
HUGGINGFACE_TOKEN = os.getenv("HF_TOKEN", "")

# Export to environment
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY
if HUGGINGFACE_TOKEN:
    os.environ["HF_TOKEN"] = HUGGINGFACE_TOKEN

GLOBAL_API_KEYS = {
    "openai": OPENAI_API_KEY,
    "gemini": GEMINI_API_KEY,
    "deepseek": DEEPSEEK_API_KEY,
    "huggingface": HUGGINGFACE_TOKEN,
}

print("=" * 60)
print("PEP-SA-RAG Configuration")
print("=" * 60)
print("BOOK_PATH:", BOOK_PATH if BOOK_PATH else "⚠️  NOT FOUND")
print("WORKDIR:", WORKDIR.resolve())
print("Global keys available:", {k: ("✓" if v else "✗") for k, v in GLOBAL_API_KEYS.items()})
print("=" * 60)

PEP-SA-RAG Configuration
BOOK_PATH: /Users/camilacruz/Documents/repositorios/ai_tutor/doctorado/libro/libro.pdf
WORKDIR: /Users/camilacruz/Documents/repositorios/ai_tutor/doctorado/libro/output/pep_sa_rag_project
Global keys available: {'openai': '✗', 'gemini': '✗', 'deepseek': '✗', 'huggingface': '✗'}


## 3. Utility functions

In [15]:

# ============================================================
# 3. Utility functions
# ============================================================
def normalize_whitespace(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def remove_page_artifacts(text: str) -> str:
    text = re.sub(r"-\n(?=\w)", "", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    return text

def clean_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = normalize_whitespace(text)
    text = remove_page_artifacts(text)
    return text.strip()

def count_words(text: str) -> int:
    return len(str(text).split())

def safe_json_dump(obj: Any, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_jsonl_write(records: List[Dict], path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def batched(iterable, n=32):
    batch = []
    for x in iterable:
        batch.append(x)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

def softmax(x, temperature=1.0):
    x = np.asarray(x, dtype=np.float64) / max(temperature, 1e-8)
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / np.sum(ex)

def lexical_tokens(text: str) -> List[str]:
    text = unidecode(str(text).lower())
    return re.findall(r"[a-z0-9]+", text)

def normalize_label(text: str) -> str:
    return re.sub(r"\s+", " ", unidecode(str(text).lower()).strip())

## 4. Step 1 — Reuse the original preprocessing pipeline

The original notebook already:
- extracts PDF pages
- cleans text
- attempts TOC-based structure
- chunks content
- builds embeddings
- creates a FAISS index

We preserve that logic and make it modular.

In [16]:

# ============================================================
# 4. PDF parsing and chunk creation
# ============================================================
import fitz
import pandas as pd
from tqdm import tqdm

def extract_pdf_pages(pdf_path: str) -> pd.DataFrame:
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        raise RuntimeError(f"Error opening PDF in extract_pdf_pages: {e}")

    rows = []
    try:
        for page_idx in tqdm(range(len(doc)), desc="Extracting pages"):
            try:
                page = doc.load_page(page_idx)
                text = page.get_text("text")
            except Exception as e:
                print(f"Warning: error reading page {page_idx + 1}: {e}")
                text = ""

            rows.append({
                "page_num": page_idx + 1,
                "raw_text": text,
            })
    finally:
        doc.close()

    return pd.DataFrame(rows)

def extract_toc(pdf_path: str) -> pd.DataFrame:
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        raise RuntimeError(f"Error opening PDF in extract_toc: {e}")

    rows = []
    try:
        toc = doc.get_toc(simple=False)
        for item in toc:
            if len(item) >= 3:
                level, title, page = item[:3]
                rows.append({
                    "level": int(level),
                    "title": str(title).strip(),
                    "page_num": int(page)
                })
    finally:
        doc.close()

    return pd.DataFrame(rows)

def build_structure_from_toc(df_pages: pd.DataFrame, df_toc: pd.DataFrame) -> pd.DataFrame:
    df = df_pages.copy()
    df["chapter_title"] = None
    df["chapter_num"] = None
    df["section_title"] = None
    df["section_num"] = None

    if len(df_toc) == 0:
        df["chapter_title"] = "book_full"
        df["chapter_num"] = 1
        return df

    toc_sorted = df_toc.sort_values(["page_num", "level"]).reset_index(drop=True)

    chapter_counter = 0
    section_counter = 0

    for i, row in toc_sorted.iterrows():
        start_page = row["page_num"]
        end_page = toc_sorted.iloc[i + 1]["page_num"] - 1 if i + 1 < len(toc_sorted) else int(df["page_num"].max())
        mask = (df["page_num"] >= start_page) & (df["page_num"] <= end_page)

        if row["level"] == 1:
            chapter_counter += 1
            section_counter = 0
            df.loc[mask, "chapter_title"] = row["title"]
            df.loc[mask, "chapter_num"] = chapter_counter

        elif row["level"] >= 2:
            if chapter_counter == 0:
                chapter_counter = 1
                df.loc[:, "chapter_title"] = df["chapter_title"].fillna("book_full")
                df.loc[:, "chapter_num"] = df["chapter_num"].fillna(1)
            section_counter += 1
            df.loc[mask, "section_title"] = row["title"]
            df.loc[mask, "section_num"] = section_counter

    df["chapter_title"] = df["chapter_title"].fillna("book_full")
    df["chapter_num"] = df["chapter_num"].fillna(1)
    return df

def chunk_text_by_words(text: str, chunk_size_words: int = 350, overlap_words: int = 80, min_chunk_words: int = 80):
    words = str(text).split()
    chunks = []
    start = 0
    chunk_id = 0

    while start < len(words):
        end = min(start + chunk_size_words, len(words))
        piece = words[start:end]
        if len(piece) >= min_chunk_words:
            chunks.append({
                "chunk_local_id": chunk_id,
                "chunk_text": " ".join(piece),
                "chunk_word_count": len(piece),
            })
            chunk_id += 1
        if end == len(words):
            break
        start = max(0, end - overlap_words)
    return chunks

def create_chunk_dataset(df_pages_struct: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["chapter_num", "chapter_title"]

    for (chapter_num, chapter_title), g in tqdm(df_pages_struct.groupby(group_cols, dropna=False), desc="Chunking chapters"):
        g = g.sort_values("page_num")
        full_text = "\n\n".join(g["clean_text"].fillna("").tolist())
        chunks = chunk_text_by_words(
            full_text,
            chunk_size_words=CHUNK_SIZE_WORDS,
            overlap_words=CHUNK_OVERLAP_WORDS,
            min_chunk_words=MIN_CHUNK_WORDS
        )
        start_page = int(g["page_num"].min())
        end_page = int(g["page_num"].max())

        for c in chunks:
            chunk_id = f"ch{int(chapter_num):02d}_chunk{int(c['chunk_local_id']):03d}"
            rows.append({
                "chunk_id": chunk_id,
                "chapter_num": int(chapter_num),
                "chapter_title": chapter_title,
                "start_page": start_page,
                "end_page": end_page,
                "chunk_local_id": c["chunk_local_id"],
                "chunk_word_count": c["chunk_word_count"],
                "text": c["chunk_text"],
                "text_for_embedding": f"Chapter {chapter_num}: {chapter_title}\n\n{c['chunk_text']}",
            })
    return pd.DataFrame(rows)

In [17]:

# ============================================================
# 5. Run preprocessing
# ============================================================
if USE_LOCAL_BOOK:
    if not Path(BOOK_PATH).exists():
        print(f"BOOK_PATH not found: {BOOK_PATH}")
        print("Update BOOK_PATH before running the preprocessing pipeline.")
    else:
        df_pages_raw = extract_pdf_pages(BOOK_PATH)
        df_pages_raw["clean_text"] = df_pages_raw["raw_text"].fillna("").apply(clean_text)
        df_pages_raw["word_count_clean"] = df_pages_raw["clean_text"].apply(count_words)
        df_pages = df_pages_raw[df_pages_raw["word_count_clean"] >= 20].copy()

        df_toc = extract_toc(BOOK_PATH)
        df_pages_struct = build_structure_from_toc(df_pages, df_toc)
        df_chunks = create_chunk_dataset(df_pages_struct)

        df_pages_struct.to_parquet(PROC_DIR / "book_pages_structured.parquet", index=False)
        df_chunks.to_parquet(PROC_DIR / "rag_chunks.parquet", index=False)
        safe_jsonl_write(df_chunks.to_dict(orient="records"), PROC_DIR / "rag_chunks.jsonl")

        print("Pages:", len(df_pages_struct))
        print("Chunks:", len(df_chunks))
        display(df_chunks.head(3))
else:
    print("Skipping local book preprocessing because USE_LOCAL_BOOK = False")

Chunking chapters: 100%|██████████| 2/2 [00:00<00:00, 77.34it/s]

Pages: 1003
Chunks: 1794


,chunk_id,chapter_num,chapter_title,start_page,end_page,chunk_local_id,chunk_word_count,text,text_for_embedding
0,ch01_chunk000,1,"Precálculo, Matemáticas para el cálculo",4,5,0,350,"S E X TA E D I C I Ó N JAMES STEWART McMASTER UNIVERSITY AND UNIVERSITY OF TORONTO LOTHAR REDLIN THE PENNSYLVANIA STATE UNIVERSITY SALEEM WATSON CALIFORNIA STATE UNIVERSITY, LONG BEACH TRADUCCIÓN:...","Chapter 1: Precálculo, Matemáticas para el cálculo\n\nS E X TA E D I C I Ó N JAMES STEWART McMASTER UNIVERSITY AND UNIVERSITY OF TORONTO LOTHAR REDLIN THE PENNSYLVANIA STATE UNIVERSITY SALEEM WATS..."
1,ch01_chunk001,1,"Precálculo, Matemáticas para el cálculo",4,5,1,122,James/Lothar Redlin y Saleem Watson Precálculo. Matemáticas para el cálculo. Sexta Edición. ISBN: 978-607-481-826-0 Visite nuestro sitio en: http://latinoamerica.cengage.com Precálculo. Matemática...,"Chapter 1: Precálculo, Matemáticas para el cálculo\n\nJames/Lothar Redlin y Saleem Watson Precálculo. Matemáticas para el cálculo. Sexta Edición. ISBN: 978-607-481-826-0 Visite nuestro sitio en: h..."
2,ch01_chunk000,1,book_full,6,1014,0,350,A C E R C A D E L O S A U T O R E S JAMES STEWART recibió su maestría de la Universidad de Stanford y su doctorado de la Universidad de Toronto. Realizó una investigación en la Universidad de Lond...,Chapter 1: book_full\n\nA C E R C A D E L O S A U T O R E S JAMES STEWART recibió su maestría de la Universidad de Stanford y su doctorado de la Universidad de Toronto. Realizó una investigación e...


## 5. Step 2 — Build dense retrieval assets

This keeps your original retrieval backbone:
- sentence embeddings
- FAISS cosine retrieval
- optional BM25

In [9]:

# ============================================================
# 6. Embeddings, BM25 and FAISS
# ============================================================
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def encode_texts(texts: List[str], batch_size: int = 32) -> np.ndarray:
    return embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

if USE_LOCAL_BOOK and 'df_chunks' in globals():
    chunk_texts = df_chunks["text_for_embedding"].tolist()
    chunk_embeddings = encode_texts(chunk_texts)

    np.save(EMB_DIR / "chunk_embeddings.npy", chunk_embeddings)
    with open(EMB_DIR / "chunk_metadata.pkl", "wb") as f:
        pickle.dump(df_chunks.to_dict(orient="records"), f)

    dim = chunk_embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(chunk_embeddings)
    faiss.write_index(faiss_index, str(INDEX_DIR / "chunk_faiss.index"))

    tokenized_corpus = [lexical_tokens(t) for t in df_chunks["text"].fillna("").tolist()]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(INDEX_DIR / "bm25.pkl", "wb") as f:
        pickle.dump(bm25, f)

    print("FAISS vectors:", faiss_index.ntotal)

Batches: 100%|██████████| 57/57 [00:17<00:00,  3.25it/s]


FAISS vectors: 1794


In [11]:

# ============================================================
# 7. Dense + BM25 retrieval helpers
# ============================================================
def load_retrieval_assets():
    index = faiss.read_index(str(INDEX_DIR / "chunk_faiss.index"))
    with open(EMB_DIR / "chunk_metadata.pkl", "rb") as f:
        metadata = pickle.load(f)
    embeddings = np.load(EMB_DIR / "chunk_embeddings.npy")
    with open(INDEX_DIR / "bm25.pkl", "rb") as f:
        bm25_obj = pickle.load(f)
    return index, metadata, embeddings, bm25_obj

def retrieve_dense(query: str, index, metadata, k: int = 5):
    q = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    scores, idxs = index.search(q, k)
    rows = []
    for s, i in zip(scores[0], idxs[0]):
        item = dict(metadata[i])
        item["score_dense"] = float(s)
        rows.append(item)
    return rows

def retrieve_bm25(query: str, metadata, bm25_obj, k: int = 5):
    toks = lexical_tokens(query)
    scores = bm25_obj.get_scores(toks)
    order = np.argsort(scores)[::-1][:k]
    rows = []
    for i in order:
        item = dict(metadata[i])
        item["score_bm25"] = float(scores[i])
        rows.append(item)
    return rows

def reciprocal_rank_fusion(result_lists: List[List[Dict]], score_key_candidates=("score_dense", "score_bm25"), k=60):
    fused = {}
    for res in result_lists:
        for rank, item in enumerate(res, start=1):
            cid = item["chunk_id"]
            fused.setdefault(cid, {"item": item, "rrf_score": 0.0})
            fused[cid]["rrf_score"] += 1.0 / (k + rank)
    out = [dict(v["item"], rrf_score=v["rrf_score"]) for v in fused.values()]
    out = sorted(out, key=lambda x: x["rrf_score"], reverse=True)
    return out

## 6. Step 3 — Extract mathematical concepts

The SA-RAG paper builds a heterogeneous graph with document nodes, entity nodes, entity-description nodes, and relation edges.  
For a math tutor, we also want **concept nodes** such as:
- linear function
- factoring
- polynomial
- slope
- quadratic equation
- completing the square

Below we implement a practical concept extraction layer. It supports two modes:

1. **Dictionary / regex-based extraction** from a concept list  
2. **LLM-assisted extraction** if you want higher recall

For reproducibility, the notebook uses the first mode by default.

In [33]:

# ============================================================
# 8. Vocabulario de conceptos (orientado a temas reales del libro)
# ============================================================
from typing import List, Dict
import re
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from unidecode import unidecode

# ------------------------------------------------------------
# Ruido frecuente en libros educativos en español
# ------------------------------------------------------------
NOISE_TERMS_ES = {
    "ejemplo", "ejemplos", "ejercicio", "ejercicios", "problema", "problemas",
    "solucion", "soluciones", "respuesta", "respuestas",
    "figura", "figuras", "tabla", "tablas", "pagina", "paginas",
    "capitulo", "seccion", "unidad", "resumen", "repaso",
    "ahora", "intente", "hacer", "considere", "suponga", "observe",
    "encuentre", "halle", "hallar", "use", "utilice", "dado",
    "cada", "cuando", "donde", "como", "entonces", "ademas",
    "mas", "menos", "dos", "tres", "cuatro", "cinco",
    "numero", "numeros", "valor", "valores",
    "punto", "puntos", "rectangulo", "vea", "note",
    "siguiente", "anteriores", "siguientes",
    "ahora intente", "intente hacer"
}

# ------------------------------------------------------------
# Conceptos semilla de apoyo en español
# ------------------------------------------------------------
MATH_SEED_TOPICS_ES = [
    "numeros reales", "intervalos", "desigualdades", "valor absoluto",
    "funciones", "dominio", "rango", "funcion lineal", "pendiente",
    "intersecciones", "graficas de funciones", "transformaciones de funciones",
    "funcion cuadratica", "polinomios", "factorizacion",
    "expresiones racionales", "funciones racionales",
    "radicales", "ecuaciones radicales", "numeros complejos",
    "funciones inversas", "composicion de funciones",
    "funciones exponenciales", "funciones logaritmicas",
    "sistemas de ecuaciones", "matrices", "determinantes",
    "sucesiones", "series", "funciones trigonometricas",
    "circulo unitario", "identidades trigonometricas", "vectores", "conicas"
]

def normalize_text_basic(text: str) -> str:
    text = str(text).strip().lower()
    text = unidecode(text)
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_heading_like_text(text: str) -> str:
    t = normalize_text_basic(text)
    t = re.sub(r"^\d+(\.\d+)*\s*", "", t)  # elimina numeración tipo 1.2.3
    t = re.sub(r"\b(objetivos|objetivo|resumen|ejercicios de repaso|practica)\b", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def is_good_concept_candidate(term: str) -> bool:
    t = clean_heading_like_text(term)

    if not t:
        return False
    if len(t) < 4:
        return False
    if t.isdigit():
        return False
    if t in NOISE_TERMS_ES:
        return False

    tokens = t.split()
    if len(tokens) > 5:
        return False
    if all(tok in NOISE_TERMS_ES for tok in tokens):
        return False
    if sum(tok in NOISE_TERMS_ES for tok in tokens) >= max(1, len(tokens) - 1):
        return False

    # elimina fragmentos raros tipo f1x2
    if re.fullmatch(r"[a-z]*\d+[a-z0-9]*", t):
        return False

    return True

def collect_structural_candidates(df_toc: pd.DataFrame = None, df_pages_struct: pd.DataFrame = None) -> List[str]:
    candidates = []

    if df_toc is not None and len(df_toc) > 0 and "title" in df_toc.columns:
        candidates.extend(df_toc["title"].dropna().astype(str).tolist())

    if df_pages_struct is not None and len(df_pages_struct) > 0:
        for col in ["chapter_title", "section_title"]:
            if col in df_pages_struct.columns:
                candidates.extend(df_pages_struct[col].dropna().astype(str).unique().tolist())

    cleaned = []
    for c in candidates:
        cc = clean_heading_like_text(c)
        if is_good_concept_candidate(cc):
            cleaned.append(cc)

    return sorted(set(cleaned))

def extract_ngram_candidates_from_chunks(
    texts: List[str],
    min_df: int = 4,
    max_df: float = 0.20,
    max_features: int = 3000
) -> pd.DataFrame:
    cleaned_texts = [normalize_text_basic(t) for t in texts if str(t).strip()]

    # stopwords básicas en español
    stopwords_es = [
        "de", "la", "el", "los", "las", "y", "o", "en", "un", "una", "unos", "unas",
        "para", "por", "con", "sin", "del", "al", "que", "se", "es", "son", "como",
        "si", "no", "a", "e", "u", "su", "sus"
    ]

    vectorizer = CountVectorizer(
        stop_words=stopwords_es,
        ngram_range=(2, 4),   # mejor frases temáticas que palabras sueltas
        min_df=min_df,
        max_df=max_df,
        max_features=max_features
    )

    X = vectorizer.fit_transform(cleaned_texts)
    vocab = vectorizer.get_feature_names_out()
    freq = X.sum(axis=0).A1

    df = pd.DataFrame({
        "candidate": vocab,
        "frequency": freq
    }).sort_values("frequency", ascending=False).reset_index(drop=True)

    df["candidate"] = df["candidate"].apply(clean_heading_like_text)
    df = df[df["candidate"].apply(is_good_concept_candidate)].copy()
    df = df.drop_duplicates(subset=["candidate"]).reset_index(drop=True)

    return df

def merge_candidate_sources(
    structural_candidates: List[str],
    df_ngram_candidates: pd.DataFrame,
    seed_topics: List[str],
    top_k_ngrams: int = 120
) -> List[str]:
    merged = []

    # 1) primero estructura del libro
    for c in structural_candidates:
        if is_good_concept_candidate(c):
            merged.append(clean_heading_like_text(c))

    # 2) luego mejores n-grams
    for c in df_ngram_candidates.head(top_k_ngrams)["candidate"].tolist():
        if is_good_concept_candidate(c):
            merged.append(clean_heading_like_text(c))

    # 3) apoyo con conceptos semilla
    for c in seed_topics:
        if is_good_concept_candidate(c):
            merged.append(clean_heading_like_text(c))

    seen = set()
    final = []
    for c in merged:
        if c and c not in seen:
            seen.add(c)
            final.append(c)

    return final

def compile_concept_patterns(concepts: List[str]) -> Dict[str, re.Pattern]:
    patterns = {}
    for c in concepts:
        escaped = re.escape(c)
        escaped = escaped.replace(r"\ ", r"\s+")
        patterns[c] = re.compile(rf"(?<!\w){escaped}(?!\w)", re.IGNORECASE)
    return patterns

def extract_concepts_from_text(text: str, concept_patterns: Dict[str, re.Pattern]) -> List[str]:
    found = []
    text = normalize_text_basic(text)
    for concept, pattern in concept_patterns.items():
        if pattern.search(text):
            found.append(concept)
    return sorted(set(found))

In [48]:
# ============================================================
# 8b. Construir candidatos de conceptos desde estructura + contenido
# ============================================================
if "df_chunks" in globals() and len(df_chunks) > 0:
    corpus_for_concepts = df_chunks["text"].fillna("").tolist()
elif "df_pages_struct" in globals() and len(df_pages_struct) > 0:
    corpus_for_concepts = df_pages_struct["clean_text"].fillna("").tolist()
else:
    raise ValueError("No hay corpus procesado disponible. Ejecuta el preprocesamiento primero.")

structural_candidates = collect_structural_candidates(
    df_toc=df_toc if "df_toc" in globals() else None,
    df_pages_struct=df_pages_struct if "df_pages_struct" in globals() else None
)

df_ngram_candidates = extract_ngram_candidates_from_chunks(
    corpus_for_concepts,
    min_df=4,
    max_df=0.20,
    max_features=3000
)

candidate_concepts = merge_candidate_sources(
    structural_candidates=structural_candidates,
    df_ngram_candidates=df_ngram_candidates,
    seed_topics=MATH_SEED_TOPICS_ES,
    top_k_ngrams=120
)

print(f"Candidatos estructurales: {len(structural_candidates)}")
print(f"Candidatos por contenido: {len(df_ngram_candidates)}")
print(f"Candidatos fusionados: {len(candidate_concepts)}")
print(candidate_concepts[:80])

Candidatos estructurales: 354
Candidatos por contenido: 2029
Candidatos fusionados: 468
['coordenadas polares', 'funciones exponenciales', 'funciones y modelos cuadraticos', 'hallar limites numerica y graficamente', 'identidades trigonometricas', 'la circunferencia unitaria', 'medida de un angulo', 'parabolas', 'que es una funcion', 'sucesiones y notacion de suma', 'vectores en dos dimensiones', 'rectas', 'el producto punto', 'elipses', 'exponentes y radicales', 'formulas de adicion y sustraccion', 'funciones polinomiales y sus graficas', 'funciones trigonometricas de numeros reales', 'graficas de ecuaciones polares', 'graficas de funciones', 'hallar limites algebraicamente', 'la funcion exponencial natural', 'sucesiones aritmeticas', 'trigonometria de triangulos rectangulos', 'division de polinomios', 'expresiones algebraicas', 'funciones logaritmicas', 'funciones trigonometricas de angulos', 'graficas trigonometricas', 'hiperbolas', 'rectas tangentes y derivadas', 'sucesiones geometr

In [46]:
# ============================================================
# 8c. Refinamiento opcional con LLM (en español)
# ============================================================
def build_concept_refinement_prompt_es(
    structural_candidates: List[str],
    ngram_candidates: List[str],
    sample_chunks: List[str],
    max_structural: int = 80,
    max_ngrams: int = 120,
    max_chunks: int = 8
) -> str:
    structural_block = "\n".join(f"- {x}" for x in structural_candidates[:max_structural])
    ngram_block = "\n".join(f"- {x}" for x in ngram_candidates[:max_ngrams])
    chunk_block = "\n\n".join(
        f"[FRAGMENTO {i+1}]\n{sample_chunks[i][:1200]}"
        for i in range(min(len(sample_chunks), max_chunks))
    )

    prompt = f"""
Estás ayudando a construir un grafo de conocimiento pedagógico a partir de un libro de precálculo en español.

Tu tarea:
1. Lee los títulos, nombres de secciones, frases candidatas y fragmentos del libro.
2. Infiere los TEMAS o CONCEPTOS matemáticos reales explicados en el libro.
3. Elimina ruido como instrucciones, ejercicios, frases genéricas y cadenas mal formadas.
4. Une duplicados y normaliza los nombres.
5. Devuelve solo una lista JSON de conceptos en español.
6. Prioriza conceptos pedagógicos reales como:
   "funciones lineales", "dominio y rango", "funciones cuadráticas",
   "funciones exponenciales", "identidades trigonométricas", etc.

CANDIDATOS ESTRUCTURALES DEL LIBRO:
{structural_block}

CANDIDATOS BASADOS EN CONTENIDO:
{ngram_block}

FRAGMENTOS DEL LIBRO:
{chunk_block}

Devuelve solo JSON:
["concepto 1", "concepto 2", "concepto 3"]
"""
    return prompt.strip()

def try_parse_json_list(raw_text: str) -> List[str]:
    raw_text = raw_text.strip()
    try:
        data = json.loads(raw_text)
        if isinstance(data, list):
            cleaned = []
            for x in data:
                cx = clean_heading_like_text(x)
                if is_good_concept_candidate(cx):
                    cleaned.append(cx)
            # quitar duplicados preservando orden
            seen = set()
            final = []
            for c in cleaned:
                if c not in seen:
                    seen.add(c)
                    final.append(c)
            return final
    except:
        pass
    return []

In [49]:
sample_chunks_for_llm = df_chunks["text"].dropna().astype(str).sample(
    min(8, len(df_chunks)), random_state=42
).tolist()

concept_prompt_es = build_concept_refinement_prompt_es(
    structural_candidates=structural_candidates,
    ngram_candidates=df_ngram_candidates["candidate"].tolist(),
    sample_chunks=sample_chunks_for_llm
)

print(concept_prompt_es[:3000])

Estás ayudando a construir un grafo de conocimiento pedagógico a partir de un libro de precálculo en español.

Tu tarea:
1. Lee los títulos, nombres de secciones, frases candidatas y fragmentos del libro.
2. Infiere los TEMAS o CONCEPTOS matemáticos reales explicados en el libro.
3. Elimina ruido como instrucciones, ejercicios, frases genéricas y cadenas mal formadas.
4. Une duplicados y normaliza los nombres.
5. Devuelve solo una lista JSON de conceptos en español.
6. Prioriza conceptos pedagógicos reales como:
   "funciones lineales", "dominio y rango", "funciones cuadráticas",
   "funciones exponenciales", "identidades trigonométricas", etc.

CANDIDATOS ESTRUCTURALES DEL LIBRO:
- 1 coordenadas polares
- 1 funciones exponenciales
- 1 funciones y modelos cuadraticos
- 1 hallar limites numerica y graficamente
- 1 identidades trigonometricas
- 1 la circunferencia unitaria
- 1 medida de un angulo
- 1 parabolas
- 1 que es una funcion
- 1 sucesiones y notacion de suma
- 1 vectores en dos d

In [50]:
# raw_llm_output = generate_with_llm(concept_prompt_es, model="gpt-4o-mini")
# refined_concepts = try_parse_json_list(raw_llm_output)

# Si el LLM devuelve una buena lista, úsala.
# Si no, usa la lista fusionada.
AUTO_MATH_CONCEPTS = refined_concepts if "refined_concepts" in globals() and len(refined_concepts) > 0 else candidate_concepts

concept_patterns = compile_concept_patterns(AUTO_MATH_CONCEPTS)

print(f"Número final de conceptos: {len(AUTO_MATH_CONCEPTS)}")
print("Muestra final:", AUTO_MATH_CONCEPTS[:60])

Número final de conceptos: 468
Muestra final: ['coordenadas polares', 'funciones exponenciales', 'funciones y modelos cuadraticos', 'hallar limites numerica y graficamente', 'identidades trigonometricas', 'la circunferencia unitaria', 'medida de un angulo', 'parabolas', 'que es una funcion', 'sucesiones y notacion de suma', 'vectores en dos dimensiones', 'rectas', 'el producto punto', 'elipses', 'exponentes y radicales', 'formulas de adicion y sustraccion', 'funciones polinomiales y sus graficas', 'funciones trigonometricas de numeros reales', 'graficas de ecuaciones polares', 'graficas de funciones', 'hallar limites algebraicamente', 'la funcion exponencial natural', 'sucesiones aritmeticas', 'trigonometria de triangulos rectangulos', 'division de polinomios', 'expresiones algebraicas', 'funciones logaritmicas', 'funciones trigonometricas de angulos', 'graficas trigonometricas', 'hiperbolas', 'rectas tangentes y derivadas', 'sucesiones geometricas', 'ceros reales de funciones polinomi

In [51]:

# ============================================================
# 9. Attach concepts to chunks
# ============================================================
if not USE_LOCAL_BOOK:
    print("USE_LOCAL_BOOK is False. Skipping concept attachment.")
elif "df_chunks" not in globals() or df_chunks is None or len(df_chunks) == 0:
    raise ValueError("df_chunks no está disponible. Ejecuta primero el preprocesamiento y la creación de chunks.")
elif "AUTO_MATH_CONCEPTS" not in globals() or len(AUTO_MATH_CONCEPTS) == 0:
    raise ValueError("AUTO_MATH_CONCEPTS no está definido o está vacío. Ejecuta primero la sección 8.")
elif "concept_patterns" not in globals() or len(concept_patterns) == 0:
    raise ValueError("concept_patterns no está definido. Ejecuta primero la sección 8.")
else:
    df_chunks = df_chunks.copy()

    def extract_concepts_from_chunk_safe(text):
        concepts = extract_concepts_from_text(text, concept_patterns)
        concepts = [c for c in concepts if isinstance(c, str) and c.strip()]
        concepts = sorted(set(concepts))
        return concepts

    df_chunks["concepts"] = df_chunks["text"].fillna("").apply(extract_concepts_from_chunk_safe)
    df_chunks["num_concepts"] = df_chunks["concepts"].apply(len)

    # Opcional: texto unido para futuras etapas de grafos/evaluación
    df_chunks["concepts_str"] = df_chunks["concepts"].apply(lambda xs: " | ".join(xs))

    print(f"Chunks totales: {len(df_chunks)}")
    print(f"Chunks con al menos 1 concepto: {(df_chunks['num_concepts'] > 0).sum()}")

    display(
        df_chunks[
            ["chunk_id", "chapter_num", "chapter_title", "num_concepts", "concepts"]
        ].head(10)
    )

Chunks totales: 1794
Chunks con al menos 1 concepto: 1729


,chunk_id,chapter_num,chapter_title,num_concepts,concepts
0,ch01_chunk000,1,"Precálculo, Matemáticas para el cálculo",3,"[precalculo matematicas para el calculo, sistemas, trabajo]"
1,ch01_chunk001,1,"Precálculo, Matemáticas para el cálculo",1,[precalculo matematicas para el calculo]
2,ch01_chunk000,1,book_full,0,[]
3,ch01_chunk001,1,book_full,28,"[ajuste lineal de datos, al estudiante, cambio promedio, capitulo 1 examen, capitulo 1 fundamentos, capitulo 2 examen, capitulo 2 funciones, combinacion de funciones, desigualdades, ecuaciones, en..."
4,ch01_chunk002,1,book_full,45,"[cambio promedio, capitulo 2 examen, capitulo 2 funciones, capitulo 3 examen, capitulo 4 examen, capitulo 5 examen, ceros reales, ceros reales de funciones polinomiales, circunferencia unitaria, c..."
5,ch01_chunk003,1,book_full,55,"[adicion y sustraccion, campos vectoriales, capitulo 5 examen, capitulo 6 examen, capitulo 7 examen, capitulo 7 trigonometria analitica, capitulo 8 examen, capitulo 9 examen, contenido, coordenada..."
6,ch01_chunk004,1,book_full,56,"[apendice, areas, campos vectoriales, capitulo 10 examen, capitulo 11 examen, capitulo 11 secciones conicas, capitulo 12 examen, capitulo 12 sucesiones y series, capitulo 13 examen, capitulo 9 exa..."
7,ch01_chunk005,1,book_full,13,"[apendice, areas, capitulo 13 examen, contenido, enfoque sobre, enfoque sobre modelado, indice, infi nito, interpretaciones de area, limites de sucesiones, sobre modelado, sucesiones, vectores]"
8,ch01_chunk006,1,book_full,19,"[capitulo 2 funciones, capitulo 7 trigonometria analitica, circunferencia unitaria, contenido, coordenadas polares, ecuaciones, ecuaciones parametricas, funcion exponencial, funciones, funciones e..."
9,ch01_chunk007,1,book_full,16,"[al estudiante, capitulo 11 secciones conicas, capitulo 7 trigonometria analitica, circunferencia unitaria, conicas, coordenadas polares, desigualdades, ecuaciones, ecuaciones parametricas, enfoqu..."


## 7. Step 4 — Infer prerequisite edges

The **PREREQ** paper formulates concept prerequisite prediction as a supervised problem using:
- latent concept representations from Pairwise-Link LDA
- a Siamese network over concept vectors

In this notebook we implement a **practical approximation** that stays faithful to the paper’s spirit:

### Representation layer
We learn **topic vectors for concepts** from the book corpus:
- chunk-term matrix
- LDA or NMF topics
- concept vector = average topic distribution of chunks that mention that concept

### Prediction layer
We build prerequisite scores using:
- chapter order prior
- concept frequency asymmetry
- co-occurrence asymmetry
- topic-space distance / similarity
- optional labeled prerequisite pairs

If labeled prerequisite data exists, we fit a supervised classifier.  
If not, we use a calibrated heuristic score.

In [52]:

# ============================================================
# 10. Topic model for concept vectors
# ============================================================
def build_topic_model(df_chunks: pd.DataFrame, n_topics: int = 40, model_kind: str = "nmf"):
    texts = df_chunks["text"].fillna("").tolist()
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9
    )
    X = vectorizer.fit_transform(texts)

    if model_kind == "lda":
        topic_model = LatentDirichletAllocation(
            n_components=n_topics,
            random_state=RANDOM_STATE,
            learning_method="batch"
        )
        Z = topic_model.fit_transform(X)
    else:
        topic_model = NMF(
            n_components=n_topics,
            random_state=RANDOM_STATE,
            init="nndsvda",
            max_iter=500
        )
        Z = topic_model.fit_transform(X)

    return vectorizer, topic_model, X, Z

def build_concept_topic_vectors(df_chunks: pd.DataFrame, Z: np.ndarray) -> pd.DataFrame:
    concept_rows = []
    for concept in sorted({c for row in df_chunks["concepts"] for c in row}):
        idx = df_chunks.index[df_chunks["concepts"].apply(lambda xs: concept in xs)].tolist()
        if not idx:
            continue
        vec = Z[idx].mean(axis=0)
        chapter_positions = df_chunks.loc[idx, "chapter_num"].astype(float).values
        concept_rows.append({
            "concept": concept,
            "support_chunks": len(idx),
            "avg_chapter": float(np.mean(chapter_positions)),
            "topic_vector": vec / (np.linalg.norm(vec) + 1e-12)
        })
    return pd.DataFrame(concept_rows)

if USE_LOCAL_BOOK and 'df_chunks' in globals():
    topic_vectorizer, topic_model, X_topic, Z_topic = build_topic_model(df_chunks, N_TOPICS, TOPIC_MODEL_KIND)
    df_concepts = build_concept_topic_vectors(df_chunks, Z_topic)
    print("Detected concepts:", len(df_concepts))
    display(df_concepts.head())

Detected concepts: 435


,concept,support_chunks,avg_chapter,topic_vector
0,a b2,12,1.0,"[0.010760938656736998, 0.40753541799655607, 0.10593475813402614, 0.18184678513232158, 0.14580552558543938, 0.014155389754165072, 0.007442319828613118, 0.0023440489267658898, 0.021372441796264238, ..."
1,adicion y sustraccion,23,1.0,"[0.0004013593271294289, 0.6281591644555874, 0.019100404940066932, 0.012716064563609646, 0.05943246465032301, 0.014447935285985147, 0.005658113010190837, 0.0030015964082930452, 0.02313573047664804,..."
2,ajustar el desplazamiento horizontal,1,1.0,"[0.0, 0.031416916067003316, 0.010573831937541944, 0.11176953371554244, 0.0, 0.0, 0.0, 0.006988345393910346, 0.002008310591218595, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7665366449679296, 0.0, 0.0, 0.0, 0..."
3,ajustar el periodo,1,1.0,"[0.0, 0.031416916067003316, 0.010573831937541944, 0.11176953371554244, 0.0, 0.0, 0.0, 0.006988345393910346, 0.002008310591218595, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7665366449679296, 0.0, 0.0, 0.0, 0..."
4,ajustar la amplitud,1,1.0,"[0.0, 0.031416916067003316, 0.010573831937541944, 0.11176953371554244, 0.0, 0.0, 0.0, 0.006988345393910346, 0.002008310591218595, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7665366449679296, 0.0, 0.0, 0.0, 0..."


In [19]:

# ============================================================
# 11. Build candidate prerequisite pairs and scores
# ============================================================
def cooccurrence_matrix(df_chunks: pd.DataFrame, concepts: List[str]) -> pd.DataFrame:
    index = {c:i for i,c in enumerate(concepts)}
    M = np.zeros((len(concepts), len(concepts)), dtype=np.float32)
    for row in df_chunks["concepts"]:
        unique = sorted(set(row))
        for a in unique:
            for b in unique:
                if a != b:
                    M[index[a], index[b]] += 1
    return pd.DataFrame(M, index=concepts, columns=concepts)

def build_prereq_features(df_chunks: pd.DataFrame, df_concepts: pd.DataFrame) -> pd.DataFrame:
    concepts = df_concepts["concept"].tolist()
    c2row = {r["concept"]: r for _, r in df_concepts.iterrows()}
    cooc = cooccurrence_matrix(df_chunks, concepts)
    chapter_map = df_concepts.set_index("concept")["avg_chapter"].to_dict()
    support_map = df_concepts.set_index("concept")["support_chunks"].to_dict()

    rows = []
    for a in concepts:
        for b in concepts:
            if a == b:
                continue
            va = np.asarray(c2row[a]["topic_vector"])
            vb = np.asarray(c2row[b]["topic_vector"])

            sim = float(np.dot(va, vb) / ((np.linalg.norm(va)+1e-12)*(np.linalg.norm(vb)+1e-12)))
            chapter_gap = float(chapter_map[b] - chapter_map[a])  # positive if a appears earlier
            support_ratio = float((support_map[a] + 1) / (support_map[b] + 1))
            cooc_ab = float(cooc.loc[a, b])
            cooc_ba = float(cooc.loc[b, a])
            asym = float((cooc_ab + 1) / (cooc_ba + 1))

            heuristic_score = (
                0.35 * sim +
                0.30 * np.tanh(chapter_gap / 3.0) +
                0.20 * np.tanh(math.log(support_ratio)) +
                0.15 * np.tanh(math.log(asym))
            )

            rows.append({
                "source_concept": a,
                "target_concept": b,
                "topic_similarity": sim,
                "chapter_gap": chapter_gap,
                "support_ratio": support_ratio,
                "cooc_asymmetry": asym,
                "heuristic_score": heuristic_score
            })
    return pd.DataFrame(rows)

if USE_LOCAL_BOOK and 'df_chunks' in globals() and 'df_concepts' in globals():
    df_prereq_features = build_prereq_features(df_chunks, df_concepts)
    display(df_prereq_features.sort_values("heuristic_score", ascending=False).head(20))

### Optional: supervised prerequisite labels

If you have a small CSV with manually labeled pairs:
- `source_concept`
- `target_concept`
- `label` (1 if source is prerequisite of target, 0 otherwise)

you can train a lightweight classifier on top of these features.

In [59]:

# ============================================================
# 12. Optional supervised prerequisite model
# ============================================================
PREREQ_LABELS_CSV = PROC_DIR / "prereq_labels.csv"   # optional

def train_prereq_classifier(df_features: pd.DataFrame, label_csv: Path):
    df_labels = pd.read_csv(label_csv)
    df = df_features.merge(df_labels, on=["source_concept", "target_concept"], how="inner")
    feats = ["topic_similarity", "chapter_gap", "support_ratio", "cooc_asymmetry", "heuristic_score"]
    X = df[feats].values
    y = df["label"].astype(int).values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]
    print("Prerequisite classifier F1:", round(f1_score(y_test, pred), 4))

    return clf, feats

if USE_LOCAL_BOOK and 'df_prereq_features' in globals():
    prereq_model = None
    prereq_features_used = None

    if PREREQ_LABELS_CSV.exists():
        prereq_model, prereq_features_used = train_prereq_classifier(df_prereq_features, PREREQ_LABELS_CSV)
    else:
        print("No labeled prerequisite CSV found. The notebook will use heuristic prerequisite scores.")

No labeled prerequisite CSV found. The notebook will use heuristic prerequisite scores.


In [65]:

# ============================================================
# 13. Final prerequisite probabilities
# ============================================================
def add_prereq_probabilities(df_prereq_features: pd.DataFrame, model=None, features_used=None):
    df = df_prereq_features.copy()
    if model is not None:
        probs = model.predict_proba(df[features_used].values)[:, 1]
        df["prereq_prob"] = probs
    else:
        # heuristic calibration to [0,1]
        scores = df["heuristic_score"].values
        probs = 1 / (1 + np.exp(-3.0 * scores))
        df["prereq_prob"] = probs
    return df

print("USE_LOCAL_BOOK:", USE_LOCAL_BOOK)
print("df_prereq_features in globals:", 'df_prereq_features' in globals())

if USE_LOCAL_BOOK and 'df_prereq_features' in globals():
    df_prereq = add_prereq_probabilities(df_prereq_features, prereq_model, prereq_features_used)
    display(df_prereq.sort_values("prereq_prob", ascending=False).head(20))
else:
    print("df_prereq not created")

USE_LOCAL_BOOK: True
df_prereq_features in globals: False
df_prereq not created


## 8. Step 5 — Build the heterogeneous graph

Inspired by the SA-RAG indexing design, we create a graph with:
- **chunk nodes**
- **concept nodes**
- optional **chapter nodes**

Edges:
- `mentions` from chunk → concept
- `same_chapter` between chunk and chapter
- `related_to` between concepts based on semantic proximity
- `prerequisite_to` between concepts from the inferred prerequisite model

This is the key bridge between the two papers:
- SA-RAG contributes the **graph retrieval**
- PREREQ contributes the **pedagogical structure**

In [63]:

# ============================================================
# 14. Build heterogeneous graph
# ============================================================
def build_heterogeneous_graph(df_chunks: pd.DataFrame, df_concepts: pd.DataFrame, df_prereq: pd.DataFrame,
                              related_sim_threshold: float = 0.35,
                              prereq_threshold: float = 0.60) -> nx.DiGraph:
    G = nx.DiGraph()

    # chapter nodes
    chapter_titles = df_chunks[["chapter_num", "chapter_title"]].drop_duplicates()
    for _, row in chapter_titles.iterrows():
        node_id = f"chapter::{int(row['chapter_num'])}"
        G.add_node(node_id, node_type="chapter", label=row["chapter_title"], chapter_num=int(row["chapter_num"]))

    # chunk nodes
    for _, row in df_chunks.iterrows():
        node_id = f"chunk::{row['chunk_id']}"
        G.add_node(
            node_id,
            node_type="chunk",
            chunk_id=row["chunk_id"],
            chapter_num=int(row["chapter_num"]),
            label=row["chunk_id"],
            text=row["text"],
            text_for_embedding=row["text_for_embedding"]
        )
        chap_id = f"chapter::{int(row['chapter_num'])}"
        G.add_edge(node_id, chap_id, edge_type="in_chapter", weight=1.0)
        G.add_edge(chap_id, node_id, edge_type="has_chunk", weight=1.0)

    # concept nodes
    concept_to_vec = {}
    for _, row in df_concepts.iterrows():
        node_id = f"concept::{row['concept']}"
        vec = np.asarray(row["topic_vector"], dtype=np.float32)
        concept_to_vec[row["concept"]] = vec
        G.add_node(
            node_id,
            node_type="concept",
            concept=row["concept"],
            label=row["concept"],
            avg_chapter=float(row["avg_chapter"]),
            support_chunks=int(row["support_chunks"]),
            topic_vector=vec.tolist()
        )

    # chunk -> concept links
    for _, row in df_chunks.iterrows():
        chunk_id = f"chunk::{row['chunk_id']}"
        for c in row["concepts"]:
            concept_id = f"concept::{c}"
            G.add_edge(chunk_id, concept_id, edge_type="mentions", weight=1.0)
            G.add_edge(concept_id, chunk_id, edge_type="mentioned_in", weight=1.0)

    # concept semantic relatedness
    concepts = df_concepts["concept"].tolist()
    for i, a in enumerate(concepts):
        va = concept_to_vec[a]
        for j, b in enumerate(concepts):
            if i >= j:
                continue
            vb = concept_to_vec[b]
            sim = float(np.dot(va, vb) / ((np.linalg.norm(va)+1e-12)*(np.linalg.norm(vb)+1e-12)))
            if sim >= related_sim_threshold:
                G.add_edge(f"concept::{a}", f"concept::{b}", edge_type="related_to", weight=sim)
                G.add_edge(f"concept::{b}", f"concept::{a}", edge_type="related_to", weight=sim)

    # prerequisite edges
    for _, row in df_prereq.iterrows():
        if row["prereq_prob"] >= prereq_threshold:
            a = f"concept::{row['source_concept']}"
            b = f"concept::{row['target_concept']}"
            G.add_edge(a, b, edge_type="prerequisite_to", weight=float(row["prereq_prob"]))

    return G

print("USE_LOCAL_BOOK:", USE_LOCAL_BOOK)
print("df_chunks in globals:", 'df_chunks' in globals())
print("df_concepts in globals:", 'df_concepts' in globals())
print("df_prereq in globals:", 'df_prereq' in globals())

if USE_LOCAL_BOOK and {'df_chunks','df_concepts','df_prereq'}.issubset(set(globals().keys())):
    G = build_heterogeneous_graph(df_chunks, df_concepts, df_prereq)
    print("Nodes:", G.number_of_nodes())
    print("Edges:", G.number_of_edges())
else:
    print("Condition not met, G not built")

USE_LOCAL_BOOK: True
df_chunks in globals: True
df_concepts in globals: True
df_prereq in globals: False
Condition not met, G not built


## 9. Step 6 — Retrieval methods to compare

We will compare several strategies:

1. **Dense RAG**
2. **Dense + BM25 fusion**
3. **Graph neighborhood expansion**
4. **SA-RAG**
5. **SA-RAG + CoT**
6. **PPR-style graph retrieval** (HippoRAG-like approximation)
7. **PEP-SA-RAG** (our proposal)

### New proposal: PEP-SA-RAG

The proposed retrieval method is:

**PEP-SA-RAG = prerequisite-aware spreading activation + population dynamics**

Its intuition is:

- a question activates a few seed concepts
- activation spreads along:
  - semantic relations
  - concept–chunk mentions
  - prerequisite relations
- activated candidates then **compete** for relevance using a population-dynamics update  
- candidates that are both:
  - semantically relevant to the query, and
  - pedagogically useful as prerequisites / bridge concepts  
  survive with higher mass

This makes the retrieval especially suitable for **mathematics education**, where understanding often depends on prerequisite chains.

In [62]:

# ============================================================
# 15. Query utilities
# ============================================================
def get_query_embedding(query: str) -> np.ndarray:
    return embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]

def score_concepts_for_query(query: str, df_concepts: pd.DataFrame) -> pd.DataFrame:
    q_emb = get_query_embedding(query)
    concept_texts = df_concepts["concept"].tolist()
    c_emb = embedding_model.encode(concept_texts, normalize_embeddings=True, convert_to_numpy=True)
    sims = c_emb @ q_emb
    out = df_concepts[["concept", "avg_chapter", "support_chunks"]].copy()
    out["query_sim"] = sims
    return out.sort_values("query_sim", ascending=False).reset_index(drop=True)

def get_seed_concepts(query: str, df_concepts: pd.DataFrame, topk: int = 8) -> List[str]:
    scores = score_concepts_for_query(query, df_concepts)
    return scores.head(topk)["concept"].tolist()

def collect_neighbor_subgraph(G: nx.DiGraph, seed_nodes: List[str], hops: int = 2) -> nx.DiGraph:
    visited = set(seed_nodes)
    frontier = set(seed_nodes)
    for _ in range(hops):
        nxt = set()
        for u in frontier:
            nxt.update(G.successors(u))
            nxt.update(G.predecessors(u))
        frontier = nxt - visited
        visited.update(frontier)
    return G.subgraph(visited).copy()

In [63]:

# ============================================================
# 16. Baseline retrievers
# ============================================================
def dense_rag_retrieve(query: str, index, metadata, k: int = 6):
    return retrieve_dense(query, index, metadata, k=k)

def dense_bm25_fusion_retrieve(query: str, index, metadata, bm25_obj, k: int = 6):
    dense = retrieve_dense(query, index, metadata, k=max(k, 10))
    sparse = retrieve_bm25(query, metadata, bm25_obj, k=max(k, 10))
    fused = reciprocal_rank_fusion([dense, sparse])
    return fused[:k]

def graph_neighbor_retrieve(query: str, G: nx.DiGraph, df_concepts: pd.DataFrame, max_docs: int = 6):
    seeds = [f"concept::{c}" for c in get_seed_concepts(query, df_concepts, TOPK_SEED_CONCEPTS)]
    subG = collect_neighbor_subgraph(G, seeds, hops=1)
    docs = []
    for n, data in subG.nodes(data=True):
        if data.get("node_type") == "chunk":
            docs.append(data)
    return docs[:max_docs]

In [64]:

# ============================================================
# 17. SA-RAG retrieval
# ============================================================
def rescale_edge_weight(w: float, c: float = 0.40) -> float:
    w = float(max(0.0, min(1.0, w)))
    return max(0.0, min(1.0, (w - c) / max(1e-12, 1 - c)))

def run_spreading_activation(G_sub: nx.DiGraph,
                             initial_nodes: List[str],
                             activation_threshold: float = 0.50,
                             edge_floor: float = 0.40) -> Dict[str, float]:
    activation = {n: 0.0 for n in G_sub.nodes()}
    for n in initial_nodes:
        if n in activation:
            activation[n] = 1.0

    queue = list(initial_nodes)
    visited = set()

    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)

        for tgt in G_sub.successors(node):
            w = G_sub[node][tgt].get("weight", 0.0)
            w = rescale_edge_weight(w, edge_floor)
            activation[tgt] = min(1.0, activation[tgt] + activation[node] * w)
            if tgt not in visited:
                queue.append(tgt)

    activated = {n: a for n, a in activation.items() if a >= activation_threshold}
    return activated

def sa_rag_retrieve(query: str, G: nx.DiGraph, df_concepts: pd.DataFrame, max_docs: int = 6):
    seed_concepts = get_seed_concepts(query, df_concepts, topk=TOPK_SEED_CONCEPTS)
    initial_nodes = [f"concept::{c}" for c in seed_concepts]
    subG = collect_neighbor_subgraph(G, initial_nodes, hops=SUBGRAPH_HOPS)

    activated = run_spreading_activation(
        subG,
        initial_nodes=initial_nodes,
        activation_threshold=SA_THRESHOLD,
        edge_floor=SA_EDGE_FLOOR
    )

    docs = []
    for n, a in sorted(activated.items(), key=lambda x: x[1], reverse=True):
        data = subG.nodes[n]
        if data.get("node_type") == "chunk":
            docs.append({**data, "activation": a})
    return docs[:max_docs], activated, subG

In [65]:

# ============================================================
# 18. HippoRAG-like approximation with Personalized PageRank
# ============================================================
def ppr_graph_retrieve(query: str, G: nx.DiGraph, df_concepts: pd.DataFrame, max_docs: int = 6):
    seed_concepts = get_seed_concepts(query, df_concepts, topk=TOPK_SEED_CONCEPTS)
    personalization = {n: 0.0 for n in G.nodes()}
    for c in seed_concepts:
        nid = f"concept::{c}"
        if nid in personalization:
            personalization[nid] = 1.0 / len(seed_concepts)

    ppr = nx.pagerank(G, alpha=0.85, personalization=personalization, weight="weight")
    docs = []
    for n, score in sorted(ppr.items(), key=lambda x: x[1], reverse=True):
        data = G.nodes[n]
        if data.get("node_type") == "chunk":
            docs.append({**data, "ppr_score": score})
    return docs[:max_docs]

## 10. Step 7 — The new method: PEP-SA-RAG

After SA activates candidate nodes, we do **population-dynamics re-ranking**.

### Why this step?

A pure spreading activation procedure may overactivate broad, common, or noisy nodes.  
In a math tutor, not every activated node is equally useful:
- some are direct answer nodes
- some are prerequisite bridge nodes
- some are generic distractors

So we add a selection mechanism inspired by **replicator dynamics**:

- each activated candidate gets a utility
- candidates with higher utility gain mass
- utility mixes:
  - query relevance
  - prerequisite support
  - path diversity / bridge value

This is the notebook’s main research contribution.

In [66]:

# ============================================================
# 19. PEP-SA-RAG: population-dynamics re-ranking
# ============================================================
def chunk_query_similarity(query: str, chunk_texts: List[str]) -> np.ndarray:
    q = get_query_embedding(query)
    c = embedding_model.encode(chunk_texts, normalize_embeddings=True, convert_to_numpy=True)
    return c @ q

def prerequisite_support_score(G_sub: nx.DiGraph, node_id: str) -> float:
    incoming = 0.0
    outgoing = 0.0
    for u, v, d in G_sub.in_edges(node_id, data=True):
        if d.get("edge_type") == "prerequisite_to":
            incoming += d.get("weight", 0.0)
    for u, v, d in G_sub.out_edges(node_id, data=True):
        if d.get("edge_type") == "prerequisite_to":
            outgoing += d.get("weight", 0.0)
    return incoming + outgoing

def bridge_score(G_sub: nx.DiGraph, node_id: str) -> float:
    try:
        UG = G_sub.to_undirected()
        bc = nx.betweenness_centrality(UG, normalized=True)
        return float(bc.get(node_id, 0.0))
    except Exception:
        return 0.0

def pep_sa_rag_retrieve(query: str, G: nx.DiGraph, df_concepts: pd.DataFrame, max_docs: int = 6):
    # 1) SA candidate generation
    docs_sa, activated, subG = sa_rag_retrieve(query, G, df_concepts, max_docs=50)

    if not docs_sa:
        return [], {}, subG, None

    # 2) candidate utilities
    doc_ids = [f"chunk::{d['chunk_id']}" for d in docs_sa]
    doc_texts = [d["text"] for d in docs_sa]
    sim = chunk_query_similarity(query, doc_texts)

    prereq_support = np.array([prerequisite_support_score(subG, nid) for nid in doc_ids], dtype=np.float32)
    bridge = np.array([bridge_score(subG, nid) for nid in doc_ids], dtype=np.float32)
    activation_mass = np.array([activated.get(nid, 0.0) for nid in doc_ids], dtype=np.float32)

    # normalize to [0,1]
    def norm(v):
        return np.zeros_like(v) if np.max(v) == np.min(v) else minmax_scale(v)

    sim_n = norm(sim)
    prereq_n = norm(prereq_support)
    bridge_n = norm(bridge)
    act_n = norm(activation_mass)

    utility = (
        0.45 * sim_n +
        0.25 * prereq_n +
        0.15 * bridge_n +
        0.15 * act_n
    )

    # 3) replicator-style update
    x = softmax(utility, temperature=POP_TEMP)
    for _ in range(POP_ITERS):
        fitness = POP_MIX_LAMBDA * utility + (1 - POP_MIX_LAMBDA) * (x @ utility)
        x = x * np.exp(fitness)
        x = x / np.sum(x)

    result = []
    for i, d in enumerate(docs_sa):
        result.append({
            **d,
            "query_sim": float(sim_n[i]),
            "prereq_support": float(prereq_n[i]),
            "bridge_score": float(bridge_n[i]),
            "activation": float(act_n[i]),
            "pep_score": float(x[i])
        })

    result = sorted(result, key=lambda z: z["pep_score"], reverse=True)[:max_docs]
    details = pd.DataFrame(result)
    return result, activated, subG, details

## 11. Step 8 — Context builder

All retrieval strategies eventually return chunks.  
This function formats them into a grounded prompt.

In [67]:

# ============================================================
# 20. Context builder
# ============================================================
def build_context_from_docs(query: str, docs: List[Dict], max_chars: int = MAX_CONTEXT_CHARS) -> str:
    parts = []
    total = 0
    for i, d in enumerate(docs, start=1):
        title = f"[Doc {i} | {d.get('chunk_id', 'unknown')} | Chapter {d.get('chapter_num', '?')}]"
        body = d.get("text", "")
        block = f"{title}\n{body}\n"
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n".join(parts)

SYSTEM_PROMPT = """You are an expert mathematics tutor and retrieval-grounded assistant.
Answer only from the provided context.
If the context is insufficient, say so explicitly.
When the user is a learner, prioritize conceptual clarity, pedagogical sequencing, and prerequisite awareness.
"""

def build_qa_prompt(question: str, context: str) -> str:
    return f"""{SYSTEM_PROMPT}

Question:
{question}

Retrieved context:
{context}

Instructions:
1. Reason step by step from the retrieved context.
2. Prefer pedagogically structured explanations.
3. Mention prerequisite concepts if they are important for understanding.
4. If the answer cannot be grounded in the context, say: "Insufficient evidence in retrieved context."

Final answer:
"""

## 12. Step 9 — Multi-LLM wrappers

This notebook is designed to compare:
- OpenAI
- Gemini
- DeepSeek
- open-source local models

The wrappers are optional.  
If an API key or GPU model is unavailable, the evaluation runner will skip that model.

In [68]:

# ============================================================
# 21. LLM wrappers
# ============================================================
@dataclass
class GenerationResult:
    provider: str
    model_name: str
    answer: str
    latency_sec: float
    success: bool
    error: Optional[str] = None

def generate_openai(prompt: str, model_name: str = OPENAI_MODEL) -> GenerationResult:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        t0 = time.time()
        resp = client.responses.create(model=model_name, input=prompt)
        txt = resp.output_text
        return GenerationResult("openai", model_name, txt, time.time() - t0, True)
    except Exception as e:
        return GenerationResult("openai", model_name, "", 0.0, False, str(e))

def generate_gemini(prompt: str, model_name: str = GEMINI_MODEL) -> GenerationResult:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(model_name)
        t0 = time.time()
        resp = model.generate_content(prompt)
        txt = resp.text
        return GenerationResult("gemini", model_name, txt, time.time() - t0, True)
    except Exception as e:
        return GenerationResult("gemini", model_name, "", 0.0, False, str(e))

def generate_deepseek(prompt: str, model_name: str = DEEPSEEK_MODEL) -> GenerationResult:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")
        t0 = time.time()
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        txt = resp.choices[0].message.content
        return GenerationResult("deepseek", model_name, txt, time.time() - t0, True)
    except Exception as e:
        return GenerationResult("deepseek", model_name, "", 0.0, False, str(e))

def generate_local_hf(prompt: str, model_name: str) -> GenerationResult:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

        t0 = time.time()
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=HUGGINGFACE_TOKEN or None)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto",
            token=HUGGINGFACE_TOKEN or None
        )
        pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
        out = pipe(prompt, max_new_tokens=350, do_sample=False)
        txt = out[0]["generated_text"][len(prompt):].strip()
        del pipe, model, tokenizer
        gc.collect()
        return GenerationResult("opensource", model_name, txt, time.time() - t0, True)
    except Exception as e:
        return GenerationResult("opensource", model_name, "", 0.0, False, str(e))

## 13. Step 10 — Retrieval pipeline runner

In [30]:

# ============================================================
# 22. Retrieval pipeline runner
# ============================================================
def run_retrieval_method(method_name: str, query: str, index, metadata, bm25_obj, G, df_concepts):
    if method_name == "dense":
        docs = dense_rag_retrieve(query, index, metadata, k=MAX_RETRIEVAL_DOCS)
        return docs, None

    elif method_name == "dense_bm25":
        docs = dense_bm25_fusion_retrieve(query, index, metadata, bm25_obj, k=MAX_RETRIEVAL_DOCS)
        return docs, None

    elif method_name == "neighbor":
        docs = graph_neighbor_retrieve(query, G, df_concepts, max_docs=MAX_RETRIEVAL_DOCS)
        return docs, None

    elif method_name == "sa_rag":
        docs, activated, subG = sa_rag_retrieve(query, G, df_concepts, max_docs=MAX_RETRIEVAL_DOCS)
        return docs, {"activated": activated, "subgraph_nodes": len(subG.nodes())}

    elif method_name == "ppr":
        docs = ppr_graph_retrieve(query, G, df_concepts, max_docs=MAX_RETRIEVAL_DOCS)
        return docs, None

    elif method_name == "pep_sa_rag":
        docs, activated, subG, details = pep_sa_rag_retrieve(query, G, df_concepts, max_docs=MAX_RETRIEVAL_DOCS)
        extra = {
            "activated": activated,
            "subgraph_nodes": len(subG.nodes()),
            "details": None if details is None else details.to_dict(orient="records")
        }
        return docs, extra

    else:
        raise ValueError(f"Unknown method: {method_name}")

def ask_with_method(question: str, method_name: str, generator_fn, generator_model_name: str,
                    index, metadata, bm25_obj, G, df_concepts):
    docs, extra = run_retrieval_method(method_name, question, index, metadata, bm25_obj, G, df_concepts)
    context = build_context_from_docs(question, docs)
    prompt = build_qa_prompt(question, context)
    result = generator_fn(prompt, generator_model_name)
    return {
        "question": question,
        "method": method_name,
        "generator_model": generator_model_name,
        "provider": result.provider,
        "success": result.success,
        "latency_sec": result.latency_sec,
        "answer": result.answer,
        "error": result.error,
        "num_docs": len(docs),
        "docs": docs,
        "extra": extra
    }

## 14. Step 11 — Custom benchmark format

For your thesis experiments, the cleanest setup is a CSV or JSONL like:

- `question`
- `gold_answer`
- `gold_supporting_chunks` (optional)
- `gold_supporting_concepts` (optional)

This lets you measure:
- answer quality
- retrieval quality
- prerequisite coverage

You can also adapt this to MuSiQue or 2WikiMultihopQA loaders later.

In [70]:

# ============================================================
# 23. Evaluation helpers
# ============================================================
def normalize_answer(s: str) -> str:
    s = normalize_label(s)
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def exact_match(pred: str, gold: str) -> float:
    return float(normalize_answer(pred) == normalize_answer(gold))

def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    common = {}
    for t in set(pred_tokens) & set(gold_tokens):
        common[t] = min(pred_tokens.count(t), gold_tokens.count(t))
    num_same = sum(common.values())
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def retrieval_hit_at_k(retrieved_docs: List[Dict], gold_chunk_ids: List[str]) -> float:
    retrieved_ids = {d.get("chunk_id") for d in retrieved_docs}
    return float(any(g in retrieved_ids for g in gold_chunk_ids))

def evaluate_predictions(df_eval: pd.DataFrame) -> pd.DataFrame:
    grouped = (df_eval
               .groupby(["provider", "generator_model", "method"], dropna=False)
               .agg(
                   n=("question", "count"),
                   avg_em=("em", "mean"),
                   avg_f1=("f1", "mean"),
                   avg_latency=("latency_sec", "mean"),
                   retrieval_hit=("retrieval_hit", "mean")
               )
               .reset_index()
               .sort_values(["avg_f1", "avg_em"], ascending=False))
    return grouped

# ============================================================
# 23. Evaluation helpers
# ============================================================
def normalize_answer(s: str) -> str:
    s = normalize_label(s)
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def exact_match(pred: str, gold: str) -> float:
    return float(normalize_answer(pred) == normalize_answer(gold))

def token_f1(pred: str, gold: str) -> float:
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    common = {}
    for t in set(pred_tokens) & set(gold_tokens):
        common[t] = min(pred_tokens.count(t), gold_tokens.count(t))
    num_same = sum(common.values())
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def retrieval_hit_at_k(retrieved_docs: List[Dict], gold_chunk_ids: List[str]) -> float:
    retrieved_ids = {d.get("chunk_id") for d in retrieved_docs}
    return float(any(g in retrieved_ids for g in gold_chunk_ids))


In [71]:
# ============================================================
# 24. Conjunto de evaluación de ejemplo para el tutor de precálculo
# ============================================================
sample_eval = [
    {
        "question": "¿Cómo se resuelve una ecuación cuadrática completando el cuadrado?",
        "gold_answer": "Se reescribe la ecuación cuadrática para que uno de los lados sea un trinomio cuadrado perfecto, luego se extrae raíz cuadrada y se despeja la incógnita.",
        "gold_supporting_chunks": []
    },
    {
        "question": "¿Qué conceptos previos debería repasar un estudiante antes de aprender logaritmos?",
        "gold_answer": "Un estudiante debería repasar exponentes, funciones exponenciales y la idea de funciones inversas.",
        "gold_supporting_chunks": []
    },
    {
        "question": "¿Cuál es la diferencia entre el dominio y el rango de una función?",
        "gold_answer": "El dominio es el conjunto de valores de entrada permitidos y el rango es el conjunto de valores de salida que produce la función.",
        "gold_supporting_chunks": []
    }
]

df_eval_questions = pd.DataFrame(sample_eval)
display(df_eval_questions)

,question,gold_answer,gold_supporting_chunks
0,¿Cómo se resuelve una ecuación cuadrática completando el cuadrado?,"Se reescribe la ecuación cuadrática para que uno de los lados sea un trinomio cuadrado perfecto, luego se extrae raíz cuadrada y se despeja la incógnita.",[]
1,¿Qué conceptos previos debería repasar un estudiante antes de aprender logaritmos?,"Un estudiante debería repasar exponentes, funciones exponenciales y la idea de funciones inversas.",[]
2,¿Cuál es la diferencia entre el dominio y el rango de una función?,El dominio es el conjunto de valores de entrada permitidos y el rango es el conjunto de valores de salida que produce la función.,[]


## 16. Step 13 — Run a small experiment

This cell compares retrieval methods while using one generator.

For large experiments, run this one model at a time.

In [31]:
# ============================================================
# 25a. Generator setup: OPEN-SOURCE only (payment APIs disabled)
# ============================================================
import time
import torch
from transformers import pipeline

# ------------------------------------------------------------
# Model registry
# ------------------------------------------------------------
# Paid / API models (disabled for now)
# OPENAI_MODEL = "gpt-4o-mini"
# GEMINI_MODEL = "gemini-1.5-flash"
# DEEPSEEK_MODEL = "deepseek-chat"

# Open-source models (enable the ones you want to test)
OPEN_SOURCE_MODELS = {
    "flan_t5_base": "distilgpt2",
    # "flan_t5_large": "google/flan-t5-large",
    # "tinyllama_chat": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # "mistral_7b_instruct": "mistralai/Mistral-7B-Instruct-v0.2",
    # "zephyr_7b_beta": "HuggingFaceH4/zephyr-7b-beta",
}

# Choose one model first for debugging
GENERATOR_MODEL_NAME = OPEN_SOURCE_MODELS["flan_t5_base"]

# ------------------------------------------------------------
# HF pipeline cache
# ------------------------------------------------------------
hf_pipeline_cache = {}

def get_hf_pipeline(model_name: str):
    if model_name not in hf_pipeline_cache:
        print(f"Loading Hugging Face model: {model_name}")

        # Heuristic task selection
        if "t5" in model_name.lower() or "flan" in model_name.lower():
            task = "text2text-generation"
        else:
            task = "text-generation"

        hf_pipeline_cache[model_name] = pipeline(
            task,
            model=model_name,
            tokenizer=model_name,
            device=0 if torch.cuda.is_available() else -1
        )
    return hf_pipeline_cache[model_name]

def generate_hf(prompt: str, model_name: str, max_new_tokens: int = 256) -> str:
    """
    Open-source text generation wrapper.
    Returns a plain string.
    """
    pipe = get_hf_pipeline(model_name)

    # Different tasks return slightly different formats
    if pipe.task == "text2text-generation":
        out = pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
        return out[0]["generated_text"]

    else:
        out = pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            return_full_text=False
        )
        return out[0]["generated_text"]

def generator_hf_wrapper(prompt: str, model_name: str):
    """
    Adapter to match the rest of the notebook output style.
    """
    start = time.time()
    try:
        answer = generate_hf(prompt=prompt, model_name=model_name, max_new_tokens=256)
        latency = time.time() - start
        return {
            "answer": answer,
            "success": True,
            "latency_sec": latency,
            "provider": "huggingface_local",
            "generator_model": model_name,
            "error": None
        }
    except Exception as e:
        latency = time.time() - start
        return {
            "answer": "",
            "success": False,
            "latency_sec": latency,
            "provider": "huggingface_local",
            "generator_model": model_name,
            "error": str(e)
        }

# ------------------------------------------------------------
# Disable payment API generators for now
# ------------------------------------------------------------
# GENERATOR_FN = generate_openai
# GENERATOR_MODEL_NAME = OPENAI_MODEL

# GENERATOR_FN = generate_gemini
# GENERATOR_MODEL_NAME = GEMINI_MODEL

# GENERATOR_FN = generate_deepseek
# GENERATOR_MODEL_NAME = DEEPSEEK_MODEL

# Active generator: OPEN-SOURCE only
GENERATOR_FN = generator_hf_wrapper

print("Open-source generator active.")
print("Selected model:", GENERATOR_MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())

Open-source generator active.
Selected model: distilgpt2
CUDA available: False


## 17. Step 14 — Compare across LLMs

Run the full benchmark after you have:
1. built the local book assets or loaded a benchmark corpus,
2. defined `df_eval_questions`,
3. enabled the generators you want through the global API keys.

At the end, use `present_final_results(...)` so the notebook saves:
- long and wide metric tables,
- bar charts for F1 / EM / hit rate / latency,
- heatmaps,
- boxplots,
- retrieval-quality vs answer-quality plots.


In [ ]:
# ============================================================
# 25a. Generator setup: OPEN-SOURCE only (robust HF version)
# ============================================================
import time
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# Paid / API models (disabled for now)
# ------------------------------------------------------------
# OPENAI_MODEL = "gpt-4o-mini"
# GEMINI_MODEL = "gemini-1.5-flash"
# DEEPSEEK_MODEL = "deepseek-chat"

# ------------------------------------------------------------
# Open-source models
# ------------------------------------------------------------
OPEN_SOURCE_MODELS = {
    "flan_t5_base": "distilgpt2",
    # "flan_t5_large": "google/flan-t5-large",
}

GENERATOR_MODEL_NAME = OPEN_SOURCE_MODELS["flan_t5_base"]

hf_model_cache = {}
hf_tokenizer_cache = {}

def get_hf_model_and_tokenizer(model_name: str):
    if model_name not in hf_model_cache:
        print(f"Loading Hugging Face model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        if torch.cuda.is_available():
            model = model.to("cuda")

        hf_tokenizer_cache[model_name] = tokenizer
        hf_model_cache[model_name] = model

    return hf_tokenizer_cache[model_name], hf_model_cache[model_name]

def generate_hf(prompt: str, model_name: str, max_new_tokens: int = 256) -> str:
    tokenizer, model = get_hf_model_and_tokenizer(model_name)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

def generator_hf_wrapper(prompt: str, model_name: str):
    start = time.time()
    try:
        answer = generate_hf(prompt=prompt, model_name=model_name, max_new_tokens=256)
        latency = time.time() - start
        return {
            "answer": answer,
            "success": True,
            "latency_sec": latency,
            "provider": "huggingface_local",
            "generator_model": model_name,
            "error": None
        }
    except Exception as e:
        latency = time.time() - start
        return {
            "answer": "",
            "success": False,
            "latency_sec": latency,
            "provider": "huggingface_local",
            "generator_model": model_name,
            "error": str(e)
        }

GENERATOR_FN = generator_hf_wrapper

print("Open-source generator active.")
print("Selected model:", GENERATOR_MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())

Open-source generator active.
Selected model: google/flan-t5-base
CUDA available: False


In [74]:
# ============================================================
# 25a.2 Inspect available retrieval functions
# ============================================================
available_fn_names = sorted([k for k, v in globals().items() if callable(v)])
retrieval_like = [x for x in available_fn_names if "retriev" in x.lower() or "rag" in x.lower() or "ppr" in x.lower() or "bm25" in x.lower() or "neighbor" in x.lower()]
print(retrieval_like)

['BM25Okapi', 'collect_neighbor_subgraph', 'dense_bm25_fusion_retrieve', 'dense_rag_retrieve', 'graph_neighbor_retrieve', 'load_retrieval_assets', 'pep_sa_rag_retrieve', 'ppr_graph_retrieve', 'retrieval_hit_at_k', 'retrieve_bm25', 'retrieve_dense', 'run_retrieval_method', 'sa_rag_retrieve']


In [64]:
# ============================================================
# 25a.1 ask_with_method (optimized: shorter context + better prompt)
# ============================================================
import time
import numpy as np

def _safe_get(result, key, default=None):
    if isinstance(result, dict):
        return result.get(key, default)
    return getattr(result, key, default)

def ask_with_method(
    question,
    method_name,
    generator_fn,
    generator_model_name,
    index,
    metadata,
    bm25_obj,
    G,
    df_concepts
):
    start_total = time.time()

    try:
        # ----------------------------------------------------
        # 1. Retrieval
        # ----------------------------------------------------
        if method_name == "dense":
            docs = retrieve_dense(
                query=question,
                index=index,
                metadata=metadata,
                k=5
            )

        elif method_name == "dense_bm25":
            docs = dense_bm25_fusion_retrieve(
                query=question,
                index=index,
                metadata=metadata,
                bm25_obj=bm25_obj,
                k=6
            )

        elif method_name == "neighbor":
            docs = graph_neighbor_retrieve(
                query=question,
                G=G,
                df_concepts=df_concepts,
                max_docs=6
            )

        elif method_name == "sa_rag":
            docs = sa_rag_retrieve(
                query=question,
                G=G,
                df_concepts=df_concepts,
                max_docs=6
            )

        elif method_name == "ppr":
            docs = ppr_graph_retrieve(
                query=question,
                G=G,
                df_concepts=df_concepts,
                max_docs=6
            )

        elif method_name == "pep_sa_rag":
            docs = pep_sa_rag_retrieve(
                query=question,
                G=G,
                df_concepts=df_concepts,
                max_docs=6
            )

        else:
            raise ValueError(f"Unknown retrieval method: {method_name}")

        if docs is None:
            docs = []

        # ----------------------------------------------------
        # 2. Build shorter context
        # ----------------------------------------------------
        context_parts = []
        for i, d in enumerate(docs):
            if isinstance(d, dict):
                text = d.get("text", "")
                chunk_id = d.get("chunk_id", f"doc_{i}")
            else:
                text = getattr(d, "text", "")
                chunk_id = getattr(d, "chunk_id", f"doc_{i}")

            context_parts.append(f"[{chunk_id}]\n{text}")

        # Usar menos contexto para acelerar y reducir ruido
        context = "\n\n".join(context_parts[:3])[:1800]

        # ----------------------------------------------------
        # 3. Better prompt for short pedagogical answers
        # ----------------------------------------------------
        prompt = f"""
Eres un tutor de matemáticas de precálculo.

Responde siempre en español.
No copies literalmente el contexto.
Da una respuesta breve, clara y pedagógica.
Resume la idea principal en 2 o 3 oraciones.
Si el contexto no es suficiente, dilo explícitamente.

Pregunta:
{question}

Contexto:
{context}

Respuesta breve y clara:
""".strip()

        # ----------------------------------------------------
        # 4. Generation
        # ----------------------------------------------------
        result = generator_fn(prompt=prompt, model_name=generator_model_name)

        answer = _safe_get(result, "answer", "")
        provider = _safe_get(result, "provider", "unknown")
        success = _safe_get(result, "success", True)
        latency_gen = _safe_get(result, "latency_sec", None)
        error = _safe_get(result, "error", None)

        total_latency = time.time() - start_total

        return {
            "question": question,
            "method": method_name,
            "generator_model": generator_model_name,
            "provider": provider,
            "success": success,
            "latency_sec": total_latency if latency_gen is None else total_latency,
            "answer": answer,
            "docs": docs,
            "num_docs": len(docs),
            "error": error
        }

    except Exception as e:
        total_latency = time.time() - start_total
        return {
            "question": question,
            "method": method_name,
            "generator_model": generator_model_name,
            "provider": "unknown",
            "success": False,
            "latency_sec": total_latency,
            "answer": "",
            "docs": [],
            "num_docs": 0,
            "error": str(e)
        }

In [76]:
# # ============================================================
# # 25a.2 Inspect retrieval function signatures
# # ============================================================
import inspect

retrieval_fns = [
    retrieve_dense,
    dense_bm25_fusion_retrieve,
    graph_neighbor_retrieve,
    sa_rag_retrieve,
    ppr_graph_retrieve,
    pep_sa_rag_retrieve
]

for fn in retrieval_fns:
    print(f"{fn.__name__}{inspect.signature(fn)}")

retrieve_dense(query: str, index, metadata, k: int = 5)
dense_bm25_fusion_retrieve(query: str, index, metadata, bm25_obj, k: int = 6)
graph_neighbor_retrieve(query: str, G: networkx.classes.digraph.DiGraph, df_concepts: pandas.core.frame.DataFrame, max_docs: int = 6)
sa_rag_retrieve(query: str, G: networkx.classes.digraph.DiGraph, df_concepts: pandas.core.frame.DataFrame, max_docs: int = 6)
ppr_graph_retrieve(query: str, G: networkx.classes.digraph.DiGraph, df_concepts: pandas.core.frame.DataFrame, max_docs: int = 6)
pep_sa_rag_retrieve(query: str, G: networkx.classes.digraph.DiGraph, df_concepts: pandas.core.frame.DataFrame, max_docs: int = 6)


In [28]:
# Define evaluation questions
sample_eval = [
    {
        "question": "¿Cómo se resuelve una ecuación cuadrática completando el cuadrado?",
        "gold_answer": "Se reescribe la ecuación cuadrática para que uno de los lados sea un trinomio cuadrado perfecto, luego se extrae raíz cuadrada y se despeja la incógnita.",
        "gold_supporting_chunks": []
    },
    {
        "question": "¿Qué conceptos previos debería repasar un estudiante antes de aprender logaritmos?",
        "gold_answer": "Un estudiante debería repasar exponentes, funciones exponenciales y la idea de funciones inversas.",
        "gold_supporting_chunks": []
    },
    {
        "question": "¿Cuál es la diferencia entre el dominio y el rango de una función?",
        "gold_answer": "El dominio es el conjunto de valores de entrada permitidos y el rango es el conjunto de valores de salida que produce la función.",
        "gold_supporting_chunks": []
    }
]

df_eval_questions = pd.DataFrame(sample_eval)
print("df_eval_questions defined with", len(df_eval_questions), "questions")

df_eval_questions defined with 3 questions


In [56]:
# ============================================================
# 25b. Debug mínimo: 1 pregunta + 1 método
# ============================================================
# Define evaluation questions
sample_eval = [
    {
        "question": "¿Cómo se resuelve una ecuación cuadrática completando el cuadrado?",
        "gold_answer": "Se reescribe la ecuación cuadrática para que uno de los lados sea un trinomio cuadrado perfecto, luego se extrae raíz cuadrada y se despeja la incógnita.",
        "gold_supporting_chunks": []
    },
    {
        "question": "¿Qué conceptos previos debería repasar un estudiante antes de aprender logaritmos?",
        "gold_answer": "Un estudiante debería repasar exponentes, funciones exponenciales y la idea de funciones inversas.",
        "gold_supporting_chunks": []
    },
    {
        "question": "¿Cuál es la diferencia entre el dominio y el rango de una función?",
        "gold_answer": "El dominio es el conjunto de valores de entrada permitidos y el rango es el conjunto de valores de salida que produce la función.",
        "gold_supporting_chunks": []
    }
]

df_eval_questions = pd.DataFrame(sample_eval)

if USE_LOCAL_BOOK and (INDEX_DIR / "chunk_faiss.index").exists():
    index, metadata, embeddings_loaded, bm25_loaded = load_retrieval_assets()

    retrieval_methods = ["dense"]
    df_eval_debug = df_eval_questions.head(1).copy()

    experiment_rows = []

    for _, row in df_eval_debug.iterrows():
        q = row["question"]
        gold = row["gold_answer"]
        gold_chunk_ids = row.get("gold_supporting_chunks", []) if isinstance(row.get("gold_supporting_chunks", []), list) else []

        print(f"Pregunta: {q}")

        for method in retrieval_methods:
            print(f"[1] Iniciando método: {method}")

            out = ask_with_method(
                question=q,
                method_name=method,
                generator_fn=GENERATOR_FN,
                generator_model_name=GENERATOR_MODEL_NAME,
                index=index,
                metadata=metadata,
                bm25_obj=bm25_loaded,
                G=G,
                df_concepts=df_concepts
            )

            print(f"[2] Método terminado: {method}")
            print("success:", out["success"])
            print("latency_sec:", out["latency_sec"])
            print("error:", out["error"])
            print("num_docs:", out["num_docs"])
            print("answer preview:", out["answer"][:300] if out["answer"] else "")

            pred = out["answer"]

            experiment_rows.append({
                "question": q,
                "gold_answer": gold,
                "provider": out["provider"],
                "generator_model": out["generator_model"],
                "method": method,
                "answer": pred,
                "success": out["success"],
                "latency_sec": out["latency_sec"],
                "em": exact_match(pred, gold) if out["success"] else 0.0,
                "f1": token_f1(pred, gold) if out["success"] else 0.0,
                "retrieval_hit": retrieval_hit_at_k(out["docs"], gold_chunk_ids) if gold_chunk_ids else np.nan,
                "num_docs": out["num_docs"],
                "error": out["error"]
            })

    df_results = pd.DataFrame(experiment_rows)
    display(df_results)

else:
    print("Retrieval assets not found yet. Run the earlier cells first.")

Pregunta: ¿Cómo se resuelve una ecuación cuadrática completando el cuadrado?
[1] Iniciando método: dense


NameError: name 'G' is not defined

In [21]:
# ============================================================
# 25b. Load assets and run evaluation (verbose + partial saves)
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

if USE_LOCAL_BOOK and (INDEX_DIR / "chunk_faiss.index").exists():
    index, metadata, embeddings_loaded, bm25_loaded = load_retrieval_assets()

    # -------------------------
    # Cambia esto para debug rápido
    # -------------------------
    retrieval_methods = [
        "dense",
        "dense_bm25",
        "neighbor",
        "sa_rag",
        "ppr",
        "pep_sa_rag"
    ]

    # Para probar rápido primero, descomenta una de estas opciones:
    # retrieval_methods = ["dense"]
    # retrieval_methods = ["dense", "dense_bm25"]
    # df_eval_run = df_eval_questions.head(1).copy()
    df_eval_run = df_eval_questions.copy()

    experiment_rows = []

    partial_results_path = Path(OUTPUT_DIR) / "pep_sa_rag_project" / "eval" / "partial_results.csv"
    partial_results_path.parent.mkdir(parents=True, exist_ok=True)

    total_runs = len(df_eval_run) * len(retrieval_methods)
    pbar = tqdm(total=total_runs, desc="Evaluating methods")

    for q_idx, (_, row) in enumerate(df_eval_run.iterrows(), start=1):
        q = row["question"]
        gold = row["gold_answer"]
        gold_chunk_ids = row.get("gold_supporting_chunks", []) if isinstance(row.get("gold_supporting_chunks", []), list) else []

        print("\n" + "=" * 80)
        print(f"Pregunta {q_idx}/{len(df_eval_run)}")
        print(q)

        for method in retrieval_methods:
            print(f" -> Ejecutando método: {method}")

            out = ask_with_method(
                question=q,
                method_name=method,
                generator_fn=GENERATOR_FN,
                generator_model_name=GENERATOR_MODEL_NAME,
                index=index,
                metadata=metadata,
                bm25_obj=bm25_loaded,
                G=G,
                df_concepts=df_concepts
            )

            pred = out["answer"]

            row_result = {
                "question": q,
                "gold_answer": gold,
                "provider": out["provider"],
                "generator_model": out["generator_model"],
                "method": method,
                "answer": pred,
                "success": out["success"],
                "latency_sec": out["latency_sec"],
                "em": exact_match(pred, gold) if out["success"] else 0.0,
                "f1": token_f1(pred, gold) if out["success"] else 0.0,
                "retrieval_hit": retrieval_hit_at_k(out["docs"], gold_chunk_ids) if gold_chunk_ids else np.nan,
                "num_docs": out["num_docs"],
                "error": out["error"]
            }

            experiment_rows.append(row_result)

            # Guardado parcial por seguridad
            pd.DataFrame(experiment_rows).to_csv(partial_results_path, index=False)

            print(f"    success={out['success']} | latency={out['latency_sec']:.2f}s | docs={out['num_docs']} | error={out['error']}")
            if out["answer"]:
                print(f"    answer preview: {out['answer'][:220].replace(chr(10), ' ')}")

            pbar.update(1)

    pbar.close()

    df_results = pd.DataFrame(experiment_rows)
    display(df_results.head())
    print(f"\nResultados parciales/finales guardados en: {partial_results_path}")

else:
    print("Retrieval assets not found yet. Run the earlier cells first.")

NameError: name 'df_eval_questions' is not defined

In [22]:
# ============================================================
# 26. Multi-LLM benchmark runner
# ============================================================
GENERATOR_SPECS = []

if OPENAI_API_KEY:
    GENERATOR_SPECS.append((generate_openai, OPENAI_MODEL))
if GEMINI_API_KEY:
    GENERATOR_SPECS.append((generate_gemini, GEMINI_MODEL))
if DEEPSEEK_API_KEY:
    GENERATOR_SPECS.append((generate_deepseek, DEEPSEEK_MODEL))

# Enable this when you have enough GPU memory and, if needed, a HF token.
ENABLE_OPEN_SOURCE_MODELS = False
if ENABLE_OPEN_SOURCE_MODELS:
    for m in OPEN_SOURCE_MODELS:
        GENERATOR_SPECS.append((generate_local_hf, m))

def run_full_benchmark(df_questions: pd.DataFrame, retrieval_methods: List[str], save_name: str = "benchmark_results.csv"):
    all_rows = []
    index, metadata, embeddings_loaded, bm25_loaded = load_retrieval_assets()

    for gen_fn, gen_model_name in GENERATOR_SPECS:
        for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc=f"Model: {gen_model_name}"):
            q = row["question"]
            gold = row["gold_answer"]
            gold_chunk_ids = row.get("gold_supporting_chunks", []) if isinstance(row.get("gold_supporting_chunks", []), list) else []

            for method in retrieval_methods:
                out = ask_with_method(
                    question=q,
                    method_name=method,
                    generator_fn=gen_fn,
                    generator_model_name=gen_model_name,
                    index=index,
                    metadata=metadata,
                    bm25_obj=bm25_loaded,
                    G=G,
                    df_concepts=df_concepts
                )
                pred = out["answer"]
                all_rows.append({
                    "question": q,
                    "gold_answer": gold,
                    "provider": out["provider"],
                    "generator_model": out["generator_model"],
                    "method": method,
                    "answer": pred,
                    "success": out["success"],
                    "latency_sec": out["latency_sec"],
                    "em": exact_match(pred, gold) if out["success"] else 0.0,
                    "f1": token_f1(pred, gold) if out["success"] else 0.0,
                    "retrieval_hit": retrieval_hit_at_k(out["docs"], gold_chunk_ids) if gold_chunk_ids else np.nan,
                    "num_docs": out["num_docs"],
                    "error": out["error"]
                })

    df_all = pd.DataFrame(all_rows)
    out_path = EVAL_DIR / save_name
    df_all.to_csv(out_path, index=False)
    print("Saved benchmark results to:", out_path)
    return df_all


## 18. Step 15 — Visualize results

In [24]:
# ============================================================
# 27. Result tables, metrics, and plots
# ============================================================
import seaborn as sns

def evaluate_predictions(df_eval: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df_eval
        .groupby(["provider", "generator_model", "method"], dropna=False)
        .agg(
            n=("question", "count"),
            success_rate=("success", "mean"),
            avg_em=("em", "mean"),
            avg_f1=("f1", "mean"),
            avg_latency=("latency_sec", "mean"),
            retrieval_hit=("retrieval_hit", "mean"),
            avg_num_docs=("num_docs", "mean")
        )
        .reset_index()
        .sort_values(["avg_f1", "avg_em", "retrieval_hit"], ascending=False)
    )
    return grouped

def save_summary_tables(df_results: pd.DataFrame, prefix: str = "results"):
    df_summary = evaluate_predictions(df_results)
    wide_f1 = df_summary.pivot_table(index=["provider", "generator_model"], columns="method", values="avg_f1")
    wide_em = df_summary.pivot_table(index=["provider", "generator_model"], columns="method", values="avg_em")
    wide_hit = df_summary.pivot_table(index=["provider", "generator_model"], columns="method", values="retrieval_hit")

    df_summary.to_csv(TABLE_DIR / f"{prefix}_summary_long.csv", index=False)
    wide_f1.to_csv(TABLE_DIR / f"{prefix}_summary_f1_wide.csv")
    wide_em.to_csv(TABLE_DIR / f"{prefix}_summary_em_wide.csv")
    wide_hit.to_csv(TABLE_DIR / f"{prefix}_summary_hit_wide.csv")

    print("Saved tables to:", TABLE_DIR)
    return df_summary, wide_f1, wide_em, wide_hit

def plot_metric_bar(df_summary: pd.DataFrame, metric: str, title: str, save_name: Optional[str] = None):
    plot_df = df_summary.copy()
    plot_df["label"] = plot_df["provider"] + " | " + plot_df["generator_model"] + " | " + plot_df["method"]
    plot_df = plot_df.sort_values(metric, ascending=True)

    plt.figure(figsize=(12, max(6, len(plot_df) * 0.35)))
    plt.barh(plot_df["label"], plot_df[metric])
    plt.xlabel(metric)
    plt.title(title)
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOT_DIR / save_name, dpi=200, bbox_inches="tight")
    plt.show()

def plot_metric_heatmap(df_summary: pd.DataFrame, metric: str = "avg_f1", save_name: Optional[str] = None):
    heat = df_summary.pivot_table(
        index="generator_model",
        columns="method",
        values=metric,
        aggfunc="mean"
    )
    plt.figure(figsize=(10, 6))
    sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlGnBu")
    plt.title(f"Heatmap of {metric}")
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOT_DIR / save_name, dpi=200, bbox_inches="tight")
    plt.show()

def plot_latency_vs_quality(df_summary: pd.DataFrame, quality_metric: str = "avg_f1", save_name: Optional[str] = None):
    plt.figure(figsize=(9, 6))
    for _, row in df_summary.iterrows():
        plt.scatter(row["avg_latency"], row[quality_metric], s=120)
        plt.text(row["avg_latency"], row[quality_metric], row["method"], fontsize=8)

    plt.xlabel("Average latency (sec)")
    plt.ylabel(quality_metric)
    plt.title(f"Latency vs {quality_metric}")
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOT_DIR / save_name, dpi=200, bbox_inches="tight")
    plt.show()

def plot_retrieval_hit_vs_f1(df_summary: pd.DataFrame, save_name: Optional[str] = None):
    plt.figure(figsize=(9, 6))
    for _, row in df_summary.iterrows():
        plt.scatter(row["retrieval_hit"], row["avg_f1"], s=120)
        plt.text(row["retrieval_hit"], row["avg_f1"], row["method"], fontsize=8)

    plt.xlabel("Retrieval hit rate")
    plt.ylabel("Average F1")
    plt.title("Retrieval quality vs answer quality")
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOT_DIR / save_name, dpi=200, bbox_inches="tight")
    plt.show()

def plot_method_boxplots(df_results: pd.DataFrame, metric: str = "f1", save_name: Optional[str] = None):
    plt.figure(figsize=(11, 6))
    sns.boxplot(data=df_results, x="method", y=metric)
    plt.xticks(rotation=30, ha="right")
    plt.title(f"Distribution of {metric} by method")
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOT_DIR / save_name, dpi=200, bbox_inches="tight")
    plt.show()

def plot_provider_comparison(df_summary: pd.DataFrame, metric: str = "avg_f1", save_name: Optional[str] = None):
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df_summary, x="method", y=metric, hue="generator_model")
    plt.xticks(rotation=30, ha="right")
    plt.title(f"{metric} by method and generator model")
    plt.tight_layout()
    if save_name:
        plt.savefig(PLOT_DIR / save_name, dpi=200, bbox_inches="tight")
    plt.show()

def present_final_results(df_results: pd.DataFrame, prefix: str = "final"):
    df_summary, wide_f1, wide_em, wide_hit = save_summary_tables(df_results, prefix=prefix)

    display(df_summary)
    display(wide_f1)
    display(wide_em)
    display(wide_hit)

    plot_metric_bar(df_summary, "avg_f1", "Average F1 by model and retrieval method", f"{prefix}_avg_f1_bar.png")
    plot_metric_bar(df_summary, "avg_em", "Average EM by model and retrieval method", f"{prefix}_avg_em_bar.png")
    plot_metric_bar(df_summary, "retrieval_hit", "Retrieval hit rate by model and retrieval method", f"{prefix}_retrieval_hit_bar.png")
    plot_metric_bar(df_summary, "avg_latency", "Average latency by model and retrieval method", f"{prefix}_latency_bar.png")

    plot_metric_heatmap(df_summary, "avg_f1", f"{prefix}_avg_f1_heatmap.png")
    plot_metric_heatmap(df_summary, "avg_em", f"{prefix}_avg_em_heatmap.png")
    plot_latency_vs_quality(df_summary, "avg_f1", f"{prefix}_latency_vs_f1.png")
    plot_retrieval_hit_vs_f1(df_summary, f"{prefix}_hit_vs_f1.png")
    plot_method_boxplots(df_results, "f1", f"{prefix}_f1_boxplot.png")
    plot_provider_comparison(df_summary, "avg_f1", f"{prefix}_provider_comparison_f1.png")

    print("Saved plots to:", PLOT_DIR)
    return df_summary

# Example:
# df_summary = present_final_results(df_results, prefix="openai_local_book")


## 19. Step 16 — Retrieval-level diagnostics

These plots are useful for the thesis because they help explain *why* a method works.

In [26]:

# ============================================================
# 28. Graph diagnostics
# ============================================================
def visualize_subgraph(subG: nx.DiGraph, max_nodes: int = 50):
    nodes = list(subG.nodes())[:max_nodes]
    H = subG.subgraph(nodes).copy()

    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(H, seed=42)

    color_map = []
    for n, d in H.nodes(data=True):
        t = d.get("node_type")
        if t == "concept":
            color_map.append("skyblue")
        elif t == "chunk":
            color_map.append("lightgreen")
        elif t == "chapter":
            color_map.append("orange")
        else:
            color_map.append("lightgray")

    nx.draw(H, pos, with_labels=True, node_size=500, font_size=8, node_color=color_map, arrows=True)
    plt.title("Retrieved subgraph")
    plt.show()

# Example:
# docs, activated, subG, details = pep_sa_rag_retrieve("How do I solve quadratic equations?", G, df_concepts, max_docs=6)
# visualize_subgraph(subG, max_nodes=40)
# display(details)

## 20. How this notebook maps to the papers

### A. Elements adapted from the SA-RAG paper
- heterogeneous graph indexing
- seed node retrieval from semantic similarity
- subgraph fetching
- spreading activation on the graph
- retrieval from activated nodes
- plug-and-play use inside RAG pipelines

### B. Elements adapted from the PREREQ paper
- concept-level modeling
- prerequisite edge prediction
- latent concept representations via topic mixtures
- using educational structure rather than only surface similarity

### C. The new contribution proposed here
**PEP-SA-RAG** adds:
- prerequisite-aware graph edges
- activation over pedagogical relations
- population-dynamics re-ranking of activated candidates
- an educational bias toward bridge concepts and prerequisites

This makes retrieval more suitable for:
- tutoring
- hint generation
- scaffolding
- adaptive sequencing in mathematics education

## 21. Recommended thesis experiment matrix

### Retrieval methods
- Dense
- Dense + BM25
- Neighbor Graph Retrieval
- PPR / HippoRAG-style approximation
- SA-RAG
- PEP-SA-RAG (proposed)

### Generators
- OpenAI
- Gemini
- DeepSeek
- Phi-4
- Llama 3.1 Instruct
- Qwen2.5 Instruct

### Evaluation axes
- EM
- token F1
- correctness (manual or LLM-as-judge, carefully validated)
- retrieval hit@k
- concept coverage
- prerequisite coverage
- latency
- context length

### Additional ablations
- without prerequisite edges
- without population dynamics
- only prerequisite edges
- only semantic relatedness edges
- different activation thresholds
- different top-k seed concepts

## 22. Notes before running large experiments

1. **The exact results of the SA-RAG paper will not be reproduced unless you use the same benchmark subsets, prompts, embeddings, and graph-construction pipeline.**
2. **The PREREQ paper uses Pairwise-Link LDA + Siamese training.**  
   This notebook uses a practical approximation that is easier to run on your domain corpus.
3. For your doctoral work, the strongest contribution is not matching the paper line by line, but showing that:
   - prerequisite-aware retrieval improves educational QA
   - bridge concepts matter in mathematical tutoring
   - population-dynamics re-ranking helps suppress noisy activations

All generated artifacts are now saved inside subfolders under `OUTPUT_DIR`, which is set to `/content/drive/MyDrive/doctorado/libro/` by default.